<a id="chapter-03"></a>

# 第 03 章　序列建模与 Transformer

[上一章](../02%20PyTorch%E4%B8%8E%E8%AE%A1%E7%AE%97%E6%9C%BA%E8%A7%86%E8%A7%89/notes.ipynb) | [学习路径](../README.md) | [下一章](../04%20%E8%A1%A5%E5%85%85%E7%AC%94%E8%AE%B0/README.md)

> **本章主线**：本章沿**RNN / LSTM → 时序预测 → 注意力 → Transformer → 大模型训练与评估**展开。图像输入和 CNN 基础见第 02 章。

**章节目录**

- [3.1 序列建模：RNN、LSTM 与时序预测](#part-03-01)
- [3.2 从序列建模到注意力](#part-03-02)
- [3.3 Token、嵌入与位置编码](#part-03-03)
- [3.4 编码器、解码器与掩码](#part-03-04)
- [3.5 训练目标与 Teacher Forcing](#part-03-05)
- [3.6 训练归一化与稳定性](#part-03-06)
- [3.7 参数与内存高效微调](#part-03-07)
- [3.8 模型压缩与剪枝](#part-03-08)
- [3.9 强化学习基础](#part-03-09)
- [3.10 AI 对齐](#part-03-10)
- [3.11 推理与评估](#part-03-11)

**阅读层级**：章标题 → `章.节` → `章.节.小节` → `章.节.小节.知识点`。每节由分隔线与学习目标开始，正文细节使用加粗标签。

**运行约定**：以当前章节为工作目录，先执行公共导入与输出路径配置，再执行目标实验的导入、数据准备和模型定义。同名变量可能在不同实验中重定义；完整训练按需运行。


In [ ]:
from pathlib import Path
OUTPUT_DIR = Path('../outputs/chapter-03')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
# 基础包
import copy
from math import * 
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

# Pytorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Variable
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

---

<a id="part-03-01"></a>

## 3.1 序列建模：RNN、LSTM 与时序预测

> **本节目标**：先理解隐藏状态和时间维度，再学习 LSTM 门控，最后完成时序预测示例。

**小节导航**

- [3.1.1 循环神经网络的定义 Definition](#sub-03-01-01)
- [3.1.2 长短期记忆单元 LSTM Long Short Term Memory Units](#sub-03-01-02)
- [3.1.3 循环神经网络实例 - 股票预测](#sub-03-01-03)

[返回章节目录](#chapter-03)



<a id="sub-03-01-01"></a>

### 3.1.1 循环神经网络的定义 Definition

循环神经网络（Recurrent Neural Network，RNN）是一类用于处理**序列数据/序列模型 Sequence Models** 的神经网络模型。它的主要特点是能够在处理序列的每个元素时，将**之前元素的信息传递给当前元素**的处理过程。

在传统的神经网络模型中，所有的输入（和输出）都是独立的。但在很多任务中，如**语音识别**或者**自然语言处理**，我们需要处理的数据是有**时间性**的，且**输入输出的长度很可能不固定**，也就是说，输出依赖于当前和之前的输入。RNN就是为了解决这类问题而提出的。

RNN的基本结构包含一个隐藏层和一个全连接层。隐藏层的特点是，它的**输出会被重新作为下一个时间步的输入**。这种反馈连接使得RNN具有一定的“记忆”能力，能够处理和“记忆”一些时序动态行为。

尽管RNN具有处理序列数据的能力，但它也存在一些问题，比如难以处理长序列数据的问题（也被称为长期依赖问题）。为了解决这个问题，研究者们提出了一些变体，如长短期记忆网络（Long Short-Term Memory，LSTM）和门控循环单元（Gated Recurrent Unit，GRU）。

总的来说，RNN是一种强大的序列模型，被广泛用于**语音识别、自然语言处理、时间序列预测等领域**。

[视频资料 吴恩达](https://www.bilibili.com/video/BV1Gb4y1k75o?p=1&vd_source=ff901644057cda4596a72384c16c4fb4)
[视频资料 木子说](https://www.bilibili.com/video/BV14a411Q7Xh?p=7&vd_source=ff901644057cda4596a72384c16c4fb4)

RNN 从结构上大概有如下**五种分类**：
<p align="center">
<img src="./images/RNN01.png" width="800" title="Fully Convolutional Neural Network Example." >
</p>

- **一对一**：固定的输入到输出，如**图像分类**；
- **一对多**：固定的输入到序列输出，如**图像文字描述**；
- **多对一**：序列输入到固定的输出，如**情感分析、情绪分类**；
- **多对多**：序列输入到序列输出，如**机器翻译**，称之为**编解码网络**；
- **同步多对多**：同步序列输入到同步序列输出，如**文本预测**，称之为**序列生成**；

将各层展开，RNN 的内部结构具体表现如下图所示
<p align="center">
<img src="./images/RNN02.jpg" width="800" title="Fully Convolutional Neural Network Example." >
</p>

- $x_t$ 表示某一时刻的输入；一个训练样本就是一整条完整的序列 $x^{(i)}=\{x^{(i)}_t\}_{t\ =\ 0}^{T_x}$ 而训练集为 $TD=\{x^{(i)}\}_{i\ =\ 0}^N$
- $o_t$ 表示某一时刻的输出 $o_t=g\ (\ Vs_t+\theta_o\ )$；一个训练样本 $x^{(i)}$ 经由RNN得到的完整输出应该为 $o^{(i)}=\{o^{(i)}_t\}_{t\ =\ 0}^{T_o}$
- $s_t$ 表示某一时刻的隐层输出 $s_t=f\ (\ Ux_t+Ws_{t-1}+\theta_s\ )=f\ (\ [U, W] [x_t\ ;\ s_{t-1}]+\theta_s\ )$，$s_0=0$
- $U/W/V$ 表示每一个神经元节点的**共享**参数，是训练的对象

RNN 的参数更新采用 [**BPTT**](https://zhuanlan.zhihu.com/p/129336512)，即 **BackPropogation Through Time** 实现。此外，RNN还有许多变种，比如可以**同时读取前后文本信息**的 **Bidirectional RNN Bi-RNN**；以及含有**多个隐藏层**的 **Stacked RNN**
<table>
    <tr>
        <td>
          <img src="./images/BIRNN.png" width="500" />
        </td>
        <td>
        </td>
        <td>
          <img src="./images/STRNN.png" width="500" /> 
    </tr>
</table>

In [ ]:
# python nn.RNN 使用示例
import torch
import torch.nn as nn
model = nn.RNN(input_size=5, hidden_size=6, num_layers=1, 
               nonlinearity='tanh', bias=True, bidirectional=False, batch_first=False)

[document](https://pytorch.org/docs/stable/generated/torch.nn.RNN.html)
- **batch_first**:  如果为True模型的输入输出格式为 (batch, seq, feature)，否则为 **<span style="color: red;">(seq, batch, feature)</span>**，默认False
- **input_size** ： 输入的特征数量，即上文中 x_t 的长度
- **hidden_size**： 隐藏层的特征数量，即上文中 s_t 的长度，也称为 RNN 的隐层节点数
- **num_layers** ： RNN的层数，默认值为1
- **nonlinearity**：隐藏层的非线性激活函数，可以选择 'tanh' 或 'relu'，默认值为 'tanh'

In [ ]:
# 初始化模型
Input = torch.randn(7, 1 ,5) # batch_first=False (seq, batch, feature)
output, s_n = model(Input)

print(output.size()) # (seq, batch, hidden_size)
print(s_n.size())    # (layer, batch, hidden_size)


<a id="sub-03-01-02"></a>

### 3.1.2 长短期记忆单元 LSTM Long Short Term Memory Units


长短期记忆网络 [Long Short-Term Memory，LSTM](https://colah.github.io/posts/2015-08-Understanding-LSTMs/) 是一种特殊的循环神经网络（RNN），它是为解决RNN在处理长序列数据时的长期依赖问题而提出的。在传统的RNN中，由于 **梯度消失（vanishing gradient）**和 **梯度爆炸（exploding gradient）**的问题，网络在训练过程中难以学习到距离当前时间步较远的信息。LSTM通过引入一个叫做 “门” 的结构，使得网络能够学习如何选择性地记住或者忘记信息，从而缓解长期依赖和梯度消失问题，但不保证完全消除梯度消失或爆炸。

具体来说，LSTM 包含四个核心概念，分别是（下述公式与演示图中，用 $\sigma$ 表示 $Sigmoid$ 函数，粉色圆圈表示 element-wise 操作）：
- 细胞状态 (cell state): 用于储存网络之前所读取的信息 $C_t$，$C_0=0$


- 遗忘门（forget gate）：决定从单元状态中丢弃多少信息 $f_t=\sigma\ (\ W_f\ [s_{t-1}\ ;\ x_{t}]+\theta_f\ )$


- 输入门（input gate） ：决定在单元状态中存储多少新信息 $i_t=\sigma\ (\ W_i\ [s_{t-1}\ ;\ x_{t}]+\theta_i\ )$，同时更新细胞状态的备选值 $\widetilde{C_t}=tanh\ (\ W_c\ [s_{t-1}\ ;\ x_{t}]+\theta_c\ )$，最后对存储之前信息的细胞状态进行最后的更新 $C_t=f_t \odot C_{t-1}+i_t \odot \widetilde{C_t}$


- 输出门（output gate）：决定单元要输出多少信息 $o_t=\sigma\ (\ W_o\ [s_{t-1}\ ;\ x_{t}]+\theta_o\ )$，$s_t=o_t \odot tanh\ (\ C_{t}\ )$，$y_t=g\ (\ Vs_t+\theta_y\ )$


这三个门的操作使得 LSTM 能够进行长期记忆和短期遗忘，因此得名长短期记忆网络。同样的，LSTM 也有 双向LSTM 与 多层LSTM 的变形。

<p align="center">
<img src="./images/LSTM01.png" width="500" title="Fully Convolutional Neural Network Example." >
</p>

In [ ]:
# python nn.LSTM 使用示例
import torch
import torch.nn as nn
model = nn.LSTM(input_size=5, hidden_size=6, num_layers=2, 
                bias=True, bidirectional=False, batch_first=False)

[document](https://pytorch.org/docs/stable/generated/torch.nn.LSTM.html)
- **batch_first**:  如果为True模型的输入输出格式为 (batch, seq, feature)，否则为 **<span style="color: red;">(seq, batch, feature)</span>**，默认False
- **input_size** ： 输入的特征数量，即上文中 x_t 的长度
- **hidden_size**： 隐藏层的特征数量，即上文中 s_t 的长度
- **num_layers** ： RNN的层数，默认值为1

In [ ]:
# 初始化模型
Input = torch.randn(7, 1 ,5) # batch_first=False (seq, batch, feature)
output, (s_n, c_n) = model(Input) # 前面一定要有括号(s_n, c_n)！

print(output.size()) # (seq, batch, hidden_size)
print(s_n.size())    # (layer, batch, hidden_size)
print(c_n.size())    # (layer, batch, hidden_size)


<a id="sub-03-01-03"></a>

### 3.1.3 循环神经网络实例 - 股票预测


- 主要使用的数据为该只股票当日的开盘价 $O_t$，最高价 $H_t$，最低价 $L_t$，收盘价 $C_t$ 与 体量 $V_t$
- 模型结构：三个串联的 LSTM 层，外加两层全连接 FNN，FNN仅有单个输出节点输出次日的股价预测
- 输入数据：(含当日）前 $T_x$ 天的收盘价与体量 $x_t^{(i)}=[\ O^{(i)}_t \ ,\ H^{(i)}_t\ ,\ L^{(i)}_t\ ,\ C^{(i)}_t\ ,\ V^{(i)}_t \ ]$，$x^{(i)}=\{x^{(i)}_t\}_{t\ =\ 1}^{T_x}$，$T_x$ 为 sequence_len 表示输入序列长度 
- 输出数据：当日后 pre_day 天的股价

比如下面的例子为，输入含当日在内的前10天数据，预测后一日的股价

> **运行提示**：该股票示例用于学习时序张量与 LSTM 调用。历史实现的标准化使用了整个序列，存在数据泄漏；不应把其误差作为严格的样本外评估。


In [ ]:
# 其他基础包
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

# 数据准备用包
import torch
import torchvision.transforms as T
from torchvision import datasets
from torch.utils.data import DataLoader

# 搭建、训练网络用包
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.autograd import Variable

In [ ]:
# 读取并保留需要的数据
pre_day, sequence_len = 1, 10 # pre_day 表示预测几天后的股价，sequence_len 为记忆天数，即 T_x
ori_data = pd.read_csv('./data/stock/stock.csv', header=0).dropna().sort_values('Date')
ori_data['label'] = ori_data.Close.shift(-pre_day)  # ori_data.Close 上移 pre_day 行作为当日的预测 label
ori_data = ori_data.dropna()

# 对数据进行标准化
data = ori_data[['Open','High','Low','Close','Volume','label']]
data = StandardScaler().fit_transform(data) # ori_data 应该是一个形如 [n_samples, n_features] 的二维数组或df
feature = data[:,:-1]
label   = data[:,-1]

# 将feature转化为 (sample_size, seq, feature)
feature = np.array([feature[ i : i + sequence_len ] for i in range(feature.shape[0]-sequence_len+1)])
label   = label[sequence_len-1 : ]
ori_data = ori_data.iloc[sequence_len-1 : ].reset_index(drop=True)

In [ ]:
# 定义一个dataloader，将训练集以batch的形式返回，详见第 02 章的 MNIST 数据加载示例
class CustomDataset(torch.utils.data.Dataset):
    def __init__(self, feature, label):
        super().__init__()
        self.features = torch.from_numpy(feature).float() # 确保.float()不然会报错
        self.labels = torch.from_numpy(label).float()

    def __getitem__(self, index):
        feture_i = self.features[index]
        label_i = self.labels[index]
        return feture_i, label_i
        
    def __len__(self):
        return len(self.labels)

In [ ]:
# 训练与测试数据准备
train_prop, sample_size = 0.8, len(label)
split_index = int(sample_size*train_prop)
train_feature, train_label = feature[:split_index], label[:split_index]
test_feature,  test_label  = feature[split_index:], label[split_index:]

train_data = CustomDataset(train_feature, train_label)
test_data  = CustomDataset(test_feature,  test_label)
all_data = CustomDataset(feature,  label)

# 构造loaders
loaders = { 'train' : torch.utils.data.DataLoader(dataset=train_data, batch_size=5, shuffle=True),
            'test'  : torch.utils.data.DataLoader(dataset=test_data, batch_size=5, shuffle=True),
            'all': torch.utils.data.DataLoader(dataset=all_data, batch_size=len(all_data), shuffle=False)} # 别打乱！

In [ ]:
# 构建网络
class LSTMFNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm_01 = nn.LSTM(input_size=feature.shape[2], hidden_size=10, num_layers=1, 
                bias=True, bidirectional=False, batch_first=True) # input (batch, seq, feature)
        
        self.lstm_02 = nn.LSTM(input_size=10, hidden_size=15, num_layers=1, 
                bias=True, bidirectional=False, batch_first=True) # input (batch, seq, feature)
        
        self.lstm_03 = nn.LSTM(input_size=15, hidden_size=10, num_layers=1, 
                bias=True, bidirectional=False, batch_first=True) # input (batch, seq, feature)
        
        self.fnn = nn.Sequential( nn.Linear(10, 10), nn.Sigmoid(), nn.Linear(10, 1) ) 


    def forward(self, x):
        x , _ = self.lstm_01(x) # (batch, seq, feature=5) -> (batch, seq, feature/hidden_size=10)
        x , _ = self.lstm_02(x) # (batch, seq, feature/hidden_size=10) -> (batch, seq, feature/hidden_size=15)
        x , (s_n, c_n) = self.lstm_03(x) # (batch, seq, feature/hidden_size=15) -> s_n (layer=1, batch, hidden_size=10) 
        s_n = s_n.squeeze() # (layer=1, batch, hidden_size=10) -> (batch, hidden_size=10)
        y = self.fnn(s_n)   # (batch, hidden_size=10) -> (batch, hidden_output=10) -> (batch, output=1)
        return y.squeeze()

In [ ]:
# 训练函数
def train(model, loaders, num_epochs, learning_rate):
    
    model.train()     ## 将模型设置为训练模式
        
    # Optimizer & Loss function
    optimizer = optim.Adam(model.parameters(), lr = learning_rate)   # 采用Adam优化器
    loss_func = nn.MSELoss()   # 回归任务采用 Regression Loss 而不是 Probabilit Loss

    for epoch in range(num_epochs):
        for i, (features, labels) in enumerate(loaders['train']):
            
            ## Forward calculation for the loss function 
            output = model(features)        # 等价于 model.forward(features)
            loss = loss_func(output, labels)

            ## Backpropagation, update parameters
            optimizer.zero_grad()           # clear gradients
            loss.backward()                 # back propgation 
            optimizer.step()                # update parameters
            
            # Display the training progress
            if (i+1) % 100 == 0:
                print ('Epoch [{}/{}], Step [{}/{}], Loss: {:.4f}' 
                       .format(epoch + 1, num_epochs, i + 1, len(loaders['train']), loss.item()))      
                         
# 测试函数                
def test(model):
    model.eval()
    test_loss = 0
    
    with torch.no_grad():    
        for features, labels in loaders['test']:
            output = model(features)
            test_loss += F.mse_loss(output.squeeze(), labels.squeeze(), reduction='sum').item() 
            
        test_loss /= len(loaders['test'].dataset)              
        print('\nTest set: Avg. loss: {:.4f}\n'.format(test_loss))

In [ ]:
# Start training
model = LSTMFNN()
train(model, loaders, num_epochs=100, learning_rate=0.01)

In [ ]:
# Save Model
torch.save(model.state_dict(), '../outputs/chapter-03/lstm.pth')

# Load Model
model = LSTMFNN()  
model.load_state_dict(torch.load('../outputs/chapter-03/lstm.pth', map_location='cpu', weights_only=True))

In [ ]:
# Test Model
test(model)

# Visualization
for features, labels in loaders['all']:
    output = model(features).detach().numpy()*ori_data.label.std() + ori_data.label.mean()
    label  = labels.numpy()*ori_data.label.std() + ori_data.label.mean()

figure = plt.figure(figsize=(15, 5));
test_index = len(loaders['train'])*loaders['train'].batch_size;
plt.plot(output[test_index:], label = 'Predict');
plt.plot(label[test_index:], label = 'True');
plt.legend();
plt.grid();

注意，在实际操作中，由于股价的波动幅度很大，为**非平稳序列**，**预测股价的意义并不大**；虽然预测**收益率**会相对来说好一点，但注意该模型在信息较少的情况下预测长期收益率的风险仍然较大。

---

<a id="part-03-02"></a>

## 3.2 从序列建模到注意力

> **本节目标**：从递归传递状态过渡到直接比较序列位置，建立自注意力与多头机制的直觉。

**小节导航**

- [3.2.1 自注意力机制 Self-attention Mechanisms](#sub-03-02-01)
- [3.2.2 Transformer - Attention is all you need](#sub-03-02-02)

[返回章节目录](#chapter-03)


Transformer

<a id="sub-03-02-01"></a>

### 3.2.1 自注意力机制 Self-attention Mechanisms


本章笔记原材料基于[李宏毅2021/2022春机器学习课程](https://www.bilibili.com/video/BV1Wv411h7kN?p=38&vd_source=ff901644057cda4596a72384c16c4fb4) & [课件](https://github.com/Fafa-DL/Lhy_Machine_Learning)，强烈建议学习该课程中的部分相关内容以加强对笔记内容的理解。


<a id="sub-03-02-01-01"></a>

#### 3.2.1.1 序列标注问题 Sequence Labeling

**任务的分类**
神经网络模型中十分常见的一类问题，是将一个向量序列 Vector Set 作为输入，然后输出一串对应的标签 Label，根据输出 Label 的个数，将常见的问题分为如下的几类：
- 每一个输入的 Vector 都有一个对应的 Label，这类模型称为 **序列标注 Sequence Labeling** 模型
- 整个 Vector Set 对应一个唯一的 Label（如第 02 章的股价预测，或者文本情感分析 Sentiment Analysis）
- 由模型自己决定生成标签的个数，这类模型称为 **Seq2Seq** 模型

<table>
    <tr>
        <td>
          <img src="./images/trs01.png" width="500" />
        </td>
        <td>
        </td>
        <td>
          <img src="./images/trs02.png" width="500" /> 
    </tr>
</table>

对于一般的 **Sequence Labeling** 问题，最基础的方法就是训练一个全连接神经网络 FC 后，将 Vector Set 中的每一个向量作为一个独立的输入，由网络得到对应的标签。然而，在某些需要**考虑上下文语境**的情境下，这种简单的模型得到的结果显然差强人意（**见上右图，两个 saw 分别为一个动词和名词，词性分析的神经网络理应得到两个不同的标签标注；但对于一个简单的全连接网络来说，这两个词化为向量后并没有本质区别**）。解决这一问题（全连接网络无法理解上下文语境）的一种手段是，将输入向量集中与目标输入向量相邻的前后若干个向量封装成一个窗口（子向量集）一并输入网络。但就算是这样的方法也是有局限的，即当**输入的向量序列大小/个数不一致**，或输出需要考虑**全文语境**时，此类方法就不能很好地处理我们的目标任务了。

读者当然也可以考虑使用前文的 **RNN** 循环网络处理此类问题，但考虑到其自带的诸多限制（比如不能并行运作，参数爆炸等），本章将将重点放在介绍另一种更先进的算法机制上，即 **自注意力机制 Self-Attention Mechanisms**。


<a id="sub-03-02-01-02"></a>

#### 3.2.1.2 自注意力机制 Self-attention Mechanisms

注意力机制不是一个神经网络结构，而是一种可以和各种神经网络（如 FC、RNN、CNN 等）结合使用的 **策略/模块**，用来改善模型的性能，使得模型可以在每一步都“关注”到与当前任务最相关的信息，或更易捕捉到当前输入的全局信息，从而有可能改善模型的性能。如下图所示，将 Vector Set 中 **所有的向量同时输入 Self-attention 层**，使得各自得到一个 **经由全局信息调整过后** 得到的 **新向量 Modified Vector**，再将这些经调整后得到的向量 **代替原向量** 输入神经网络（如 FC、RNN、CNN 等）得到最终的输出；在同一个神经网络建模中，也可以叠加 **多个 Self-attention 层** 随时关注与当前任务最相关的新信息。**Self-attention Mechanisms** 的机制最终为 [**Transformer**](https://arxiv.org/abs/1706.03762) 的提出奠定了理论基础。

<table>
    <tr>
        <td>
          <img src="./images/trs03.png" width="500" />
        </td>
        <td>
        </td>
        <td>
          <img src="./images/trs04.png" width="500" /> 
    </tr>
</table>

Self-attention 模块大致可以拆分成如下几个步骤：

- 两两计算输入向量 $(\ a^i\ ,\ a^j\ )\subseteq\{a^n\}_{n=1}^N$ 之间的**关联参数 Attention Scores** $\alpha_{i,j}:j=1\cdots N$     

 - 计算向量 $a^i$ 各自的 **query** 向量 $q^i=W^qa^i$（此处的上角标均为上标不是指数）和 **key** 向量 $k^i=W^ka^i$；其中 $W^q$ 与 $W^k$ 由学习得到
 - 计算 Attention Score $a_{i,j}$，常见的有以下三种算法：$$a_{i,j}=q^i \cdot k^j\ \ \ or\ \ \ a_{i,j}=\frac{(q^i \cdot k^j)}{\sqrt{d_k}}\ \ \ or\ \ \ a_{i,j}=\beta \cdot tanh(\ q^i + k^j\ )$$ 其中 $d_k$ 为 $q^i$ 与 $k^j$ 的长度，$\beta$ 由学习得到；第二种算法称为 **Scaled Dot-Product Attention**，也是 **Pytorch** 的内置算法
 - 进一步使用 **Softmax** 函数将 Attention Scores 化为 weights $\{\alpha_{i,j}\}_{j=1}^N \overset{Softmax}{\longrightarrow}  \{\alpha_{i,j}'\}_{j=1}^N$

<table>
    <tr>
        <td>
          <img src="./images/trs05.png" width="500" />
        </td>
        <td>
        </td>
        <td>
          <img src="./images/trs06.png" width="500" /> 
    </tr>
</table>



- 提取输入向量各自的信息后，加权输出 **Modified Vector Set** $\{b^n\}_{n=1}^N$
 - 计算向量 $a^i$ 各自的提取信息 $v^i=W^va^i$；其中 $W^v$ 由学习得到
 - 由 $\{\alpha_{i,j}'\}_{j=1}^N$ 对 $\{a^n\}_{n=1}^N$ 加权得到 $a^i$ 的提取信息 **Extract Information**，即 $a^i$ 对应的 modified vector $b^i$ $$b^i=a'_{i,\ \bullet }\ v^\bullet = {\textstyle \sum_{j=1}^{N}}\  a'_{i,j}\ v^j$$
 
 <table>
    <tr>
        <td>
          <img src="./images/trs07.png" width="500" />
        </td>
        <td>
        </td>
        <td>
          <img src="./images/trs08.png" width="500" /> 
    </tr>
</table>

总结成如下的 **matrix notations**
 <table>
    <tr>
        <td>
          <img src="./images/trs09.png" width="500" />
        </td>
        <td>
        </td>
        <td>
          <img src="./images/trs10.png" width="500" /> 
    </tr>
</table>
 <table>
    <tr>
        <td>
          <img src="./images/trs11.png" width="500" />
        </td>
        <td>
        </td>
        <td>
          <img src="./images/trs12.png" width="500" /> 
    </tr>
</table>


<a id="sub-03-02-01-03"></a>

#### 3.2.1.3 多头自注意力机制 - 捕捉更多的特征 Multi-head Self-attention

自注意力机制的一种拓展称为 **多头自注意力机制 Multi-head Self-attention**，它通过引入更多的参数 **计算多组 $(\ q^{i,j}\ ,\ k^{i,j}\ ,\ v^{i,j}\ )$** 从而实现了对输入向量更多特征的捕捉，示意图如下（ **num_heads = 2** ）：

- 先对输入向量各做三种单次线性变换 $(\ q^{i}\ ,\ k^{i}\ ,\ v^{i}\ )\ =\ (\ W^qa^i\ ,\ W^ka^i\ ,\ W^va^i\ )\ \ s.t.\ \ i=1\cdots N$ 分别为初始的 query，key，extracted info

- 对每一个 $i$ 与 $h=head\_num$ 计算 $(\ q^{i}\ ,\ k^{i}\ ,\ v^{i}\ )\ \longrightarrow\ (\ \{q^{i,t}\}_{t=1}^h\ ,\ \{k^{i,t}\}_{t=1}^h\ ,\ \{v^{i,t}\}_{t=1}^h\ )\ \ \ s.t.\ \ \ q^{i,t}=W^{q,t}q^i,\ \ k^{i,t}=W^{k,t}k^i,\ \ v^{i,t}=W^{v,t}v^i$
- 固定 $t$，将 $(\ \{q^{i,t}\}_{i=1}^N\ ,\ \{k^{i,t}\}_{i=1}^N\ ,\ \{v^{i,t}\}_{i=1}^N\ )$ 视为一套参数，重复单头自注意力的计算得到 $\{b^{i,t}\}_{i=1}^N\ \ s.t.\ \ t=1\cdots h$
- 固定 $i$，训练一个新参数矩阵 $W^o$ 将 $[b^{i,1}; \cdots;b^{i,t};\cdots;b^{i,h}]$ 映射回原来的维度$$b^i=W^o\ [b^{i,1}; \cdots;b^{i,t};\cdots;b^{i,h}]\ \ s.t.\ \ i=1\cdots N$$

 <table>
    <tr>
        <td>
          <img src="./images/trs13.png" width="500" />
        </td>
        <td>
        </td>
        <td>
          <img src="./images/trs14.png" width="500" /> 
    </tr>
</table>

下图展示了 [**Pytorch**](https://pytorch.org/docs/stable/generated/torch.nn.MultiheadAttention.html) 中多头自注意力的计算方式：
<p align="center">
<img src="./images/trs15.png" width="300" title="Fully Convolutional Neural Network Example." >
</p>

**torch.nn.MultiheadAttention(embed_dim, num_heads, dropout=0.0, bias=True, add_bias_kv=False, add_zero_attn=False, kdim=None, vdim=None, batch_first=False, device=None, dtype=None)**
- embed_dim: Vector_length d_k.
- num_heads: Number of parallel attention heads. Note that embed_dim will be split across num_heads (i.e. each head will have dimension embed_dim // num_heads).
- batch_first: If True, then the input and output tensors are provided as (batch, seq, feature); Default: False (seq, batch, feature).
- [More detailed interpretation](https://blog.csdn.net/weixin_45581089/article/details/126530111)


<a id="sub-03-02-01-04"></a>

#### 3.2.1.4 位置编码 - 捕捉位置信息 Positional Encoding

经过之前的讨论，不难发现自注意力机制之余 RNN 忽略了对输入向量组**位置信息**的考虑。为解决这一问题，引进了 **Position Encoding** 技术。即在将向量组中的向量输入 **Self-attention Block** 之前，给他们依照自己在向量组中的位置，加上一个包含其位置信息的 **位置向量 Positional Vector**，即下图 $e^i$。$e^i$ 的形式是**人为规定的**或**通过某种规则产生的**，在最早的 [**Attention is all you need**](https://arxiv.org/abs/1706.03762) 这篇文章中，采用的位置矢量如下左图所示。实际上，如何产生合适的位置矢量至今仍是一有待研究的开放问题，目前常用的方法在 [这篇文章 (2020，03)](https://arxiv.org/abs/2003.09229) 中有详细论述，见下右图。

 <table>
    <tr>
        <td>
          <img src="./images/trs16.png" width="500" />
        </td>
        <td>
        </td>
        <td>
          <img src="./images/trs17.png" width="500" /> 
    </tr>
</table>



<a id="sub-03-02-01-05"></a>

#### 3.2.1.5 自注意力机制与神经网络

- CNN is simplified self-attention / Self-attention is the complex version of CNN (view pixels as vectores whose len = num_channels); see [this paper](https://arxiv.org/abs/1911.03584) for more details.
- CNN works better for less data while Self-attention works better for more data; see [this paper](https://arxiv.org/pdf/2010.11929.pdf) for more details.

<p align="center">
<img src="./images/trs18.png" width="800" title="Fully Convolutional Neural Network Example." >
</p>


- The outputs of the RNN are nonparallel, while the outputs of the Self-attention are parallel; 
- Self-attention 加上若干调整后也可以得到 RNN; see [this paper](https://arxiv.org/abs/2006.16236) for more details.


 <table>
    <tr>
        <td>
          <img src="./images/trs19.png" width="500" />
        </td>
        <td>
        </td>
        <td>
          <img src="./images/trs20.png" width="500" /> 
    </tr>
</table>



<a id="sub-03-02-02"></a>

### 3.2.2 Transformer - Attention is all you need

Transformer 是一个 **Sequence-to-sequence (Seq2seq)** 的模型，回顾本节“序列标注问题”中的定义，将一个向量序列 Vector Set 作为输入，然后输出一串对应的标签，由模型自己决定生成标签的个数，这类模型称为 **Seq2Seq** 模型。此类模型被广泛应用于**语音识别、机器翻译、语音翻译**的任务中。

---

<a id="part-03-03"></a>

## 3.3 Token、嵌入与位置编码

> **本节目标**：把 token ID 转为 embedding，并显式加入位置信息。

**小节导航**

- [3.3.1 Token 化与输入表示](#sub-03-03-01)
- [3.3.2 Embedding 与位置编码：代码实现](#sub-03-03-02)

[返回章节目录](#chapter-03)


<div align="center">
<img src="./images/IM01model.png" width=40% height=40%/>
</div>


<a id="sub-03-03-01"></a>

### 3.3.1 Token 化与输入表示


- 对 input **token 化**（下图一），指将句子拆分成token（可为词、子词、字符或字节片段），每个 token 在 token 表中对应一个数字 label；因此 token 化后，句子被转换成一个一维向量（token ID） $\mathbf{x}\in\mathbb{R}^{n_1}$；
- 对 $\mathbf{x}\in\mathbb{R}^{n_1}$ 进行**词嵌入 embedding**（下图二），指将 $\mathbf{x}$ 中的每一个元素（对应原句子中的一个 token label）都映射到更高维的空间 $\mathbb{R}^{n_2}$，从而得到 $n_1$ 个 $\mathbb{R}^{n_2}$ 中的向量，即 $\mathbf{x}\in\mathbb{R}^{n_1}\to\mathbf{A}\in\mathbb{R}^{n_1\times n_2}$；这一步可以通过调用 torch.nn 中的 nn.Embedding 及以下步骤实现，假设 $n_2$ 为偶数：
    - **embedding_layer = nn.Embedding(num_embeddings, embedding_dim=$n_2$)**，num_embeddings 为 token 表的大小；
    - **$\mathbf{A}$ = embedding_layer($\mathbf{x}$)**
- 对 $\mathbf{A}\in\mathbb{R}^{n_1\times n_2}$ 中各个**行向量（词向量）**$a^i\in\mathbb{R}^{n_2}$ 进行**位置编码 positional encoding**（下图三）。即对每一个 $a^i$，依照其在向量组中的位置，加上一个包含其位置信息的**位置向量 positional vector**，即 $e^i:=\mathbf{E}[i,:]\in\mathbb{R}^{n_2}$，其中 $\mathbf{E}\in\mathbb{R}^{N_1\times n_2}$ 是人为规定产生的，$N_1$ 是模型可处理的最长句子的长度。在 [**Attention is all you need**](https://arxiv.org/abs/1706.03762) 中采用的位置矢量如下图三所示。实际上，如何产生合适的位置矢量至今仍是一有待研究的开放问题，目前常用的方法在 [这篇文章](https://arxiv.org/abs/2003.09229) 中有详细论述，见下图四。经过位置编码后 $\mathbf{A}\in\mathbb{R}^{n_1\times n_2}\to\mathbf{A}_{pos}:=\mathbf{A}+\mathbf{E}[1:n_1,:]\in\mathbb{R}^{n_1\times n_2}$。

<p float="left">
  <img src="./images/IM02token.png" alt="Image 1" width="24%"  height="150"/>
  <img src="./images/IM03embed.png" alt="Image 2" width="24%"  height="150"/>
    <img src="./images/trs16.png" alt="Image 4" width="24%"  height="150"/>
    <img src="./images/trs17.png" alt="Image 5" width="24%"  height="150"/>
</p>


<a id="sub-03-03-02"></a>

### 3.3.2 Embedding 与位置编码：代码实现



In [ ]:
# 定义 Embedding 层
class Embeddings(nn.Module): 
    
    def __init__(self, num_embeddings, embedding_dim, device='cpu'):
        
        # super(A, self).__init__() 被用来调用A的父类（也就是nn.Module）的初始化函数
        super(Embeddings, self).__init__()
        self.embedding_dim = embedding_dim
        self.embedding_layer = nn.Embedding(num_embeddings, embedding_dim, device=device)
        self.device = device
    
    def forward(self, x):          
        # x为一个大小为 [batch_size, n_1] 的二维张量，代表 token_id
        # 乘上 sqrt(self.embedding_dim) 指对嵌入向量进行一定程度上的放缩
        x = self.embedding_layer(x.to(self.embedding_layer.weight.device)) * sqrt(self.embedding_dim)
        return x.to(self.device)

# 试运行
embedding_dim = 500
embedding = Embeddings(2000, embedding_dim, device=device)    # 词库大小 2000，嵌入维度 500
x = torch.tensor([[1,4,6], [2,1,1900]])                       # 两个 token_id 向量 2*3，代表两句话，每句话 3 个 token
A = embedding(x)                                              # 输出两个 token_id 向量各自对应的 embedding 2*3*500，同时将 A 转移至 gpu
print(A)

In [ ]:
# 定义 Positional Encoding 层（sinusoid）
class PositionalEncoding(nn.Module):
    
    def __init__(self, embedding_dim, max_len=5000, device='cpu'):

        # embedding_dim: 词嵌入维度，也即模型维度，必须为偶数
        # max_len: 模型可处理的最长句子的长度（ N_1 ）
        # device: 运行位置 CPU 或 GPU
        
        super(PositionalEncoding, self).__init__()

        if embedding_dim <= 0 or embedding_dim % 2:
            raise ValueError("embedding_dim must be a positive even integer")
        # 构造位置编码矩阵 E (pe)
        pe = torch.zeros(max_len, embedding_dim, device=device, requires_grad = False) # N_1 * n_2
        pos = torch.arange(0, max_len, device=device).float().unsqueeze(dim=1)   # N_1 * 1 : [[0]; [1]; ...; [N_1]] 
        _2i = torch.arange(0, embedding_dim, step=2, device=device).float()      # n_2/2 : [0, 2, 4, ..., (n_2+1)/2]
        pos2i = pos / (10000 ** (_2i / embedding_dim))                           # N_1 * n_2/2 : [0, 2, 4, ..., (n_2+1)/2]
        pe[:, 0::2] = torch.sin(pos2i)
        pe[:, 1::2] = torch.cos(pos2i)
        pe = pe.unsqueeze(0) # N_1 * n_2/2 : [0, 2, 4, ..., (n_2+1)/2]
        self.register_buffer('pe', pe) # 注册成模型的 buffer，表示除超参数和一般参数类的另一种参数，不需要随着优化步骤进行更新，且再次调用保存的模型时避免二次计算
        
    def forward(self, A):
        return A + self.pe[:,:A.size(1),:]
   

# 试运行
positional_encoding = PositionalEncoding(embedding_dim, device=device) 
A_encoded = positional_encoding(A)
print(A_encoded)

import seaborn as sns
sns.heatmap(positional_encoding.pe[0].cpu().numpy())

关于 [**self.register_buffer**](https://blog.csdn.net/qq_43391414/article/details/121060866) 的详细解释。

---

<a id="part-03-04"></a>

## 3.4 编码器、解码器与掩码

> **本节目标**：沿注意力 → 归一化 → 子层 → 编码器/解码器 → 完整模型的顺序阅读实现。

**小节导航**

- [3.4.1 编码器与解码器：整体机制](#sub-03-04-01)
- [3.4.2 注意力计算与因果掩码](#sub-03-04-02)
- [3.4.3 LayerNorm：定义与数值验证](#sub-03-04-03)
- [3.4.4 多头注意力、前馈网络与残差子层](#sub-03-04-04)
- [3.4.5 编码器的堆叠](#sub-03-04-05)
- [3.4.6 解码器与交叉注意力](#sub-03-04-06)
- [3.4.7 输出层、模型整合与前向验证](#sub-03-04-07)

[返回章节目录](#chapter-03)



<a id="sub-03-04-01"></a>

### 3.4.1 编码器与解码器：整体机制


注意力机制使得模型可以在每一步都关注到与当前任务最相关的信息，或更易捕捉到当前输入的全局信息，从而有可能改善模型的性能。如下图所示，将 $\mathbf{A}_{pos}$ 中**每个（行）向量（各自代表之前的一个 token）同时输入 attention 层**，使得各自得到一个经由全局信息调整过后得到的新向量，再将这些经调整后得到的新向量代替原向量输入神经网络（如 FC）得到该 attention block 的输出。

<table>
    <tr>
        <td>
          <img src="./images/trs03.png" width="400" />
        </td>
        <td>
        </td>
        <td>
          <img src="./images/trs04.png" width="400" /> 
    </tr>
</table>

记 $\mathbf{A}_{pos}$ 中行向量为 $\{a^i\}_{i=1}^{n_1}$（此处的上角标均为上标不是指数），(self)attention 模块大致可以拆分成如下几个步骤：

- 计算向量 $a^i$ 各自的 **query** 向量 $q^i=a^iW^q$ 和 **key** 向量 $k^i=a^iW^k$；其中 $W^q$ 与 $W^k$ 由学习得到；
- 计算 attention score $\alpha_{i,j}$，常见的有以下三种算法（其中 $d_k$ 为 $q^i$ 与 $k^j$ 的长度，$\beta$ 由学习得到；第二种算法称为 **scaled dot-product attention**，也是 **Pytorch** 的内置算法）：$$a_{i,j}=q^i \cdot k^j\ \ \ \mathrm{or}\ \ \ a_{i,j}=\frac{(q^i \cdot k^j)}{\sqrt{d_k}}\ \ \ \mathrm{or}\ \ \ a_{i,j}=\beta \cdot tanh(\ q^i + k^j\ ).$$ 
- 对每一个 $i$ 进一步使用 **Softmax** 函数将 attention scores 化为 weights $\{\alpha_{i,j}\}_{j=1}^{n_1} \overset{Softmax}{\longrightarrow}  \{\alpha_{i,j}'\}_{j=1}^{n_1}$；
- 计算向量 $a^i$ 各自的提取信息 $v^i=a^iW^v$，其中 $W^v$ 由学习得到；
- 由 $\{\alpha_{i,j}'\}_{j=1}^{n_1}$ 对 $\{v^i\}_{i=1}^{n_1}$ 加权得到 $a^i$ 的提取信息 **extract information**，即 $a^i$ 对应的 modified vector $b^i$：
  $$b^i=\alpha'_{i,\ \bullet }\ v^\bullet = {\textstyle \sum_{j=1}^{n_1}}\ \alpha'_{i,j}\ v^j$$

总结成如下的 **matrix notations**，有:
- $\mathbf{Q}=\mathbf{A}_{pos}W^q=[q^1;\cdots;q^{n_1}]$，$\mathbf{K}=\mathbf{A}_{pos}W^k=[k^1;\cdots;k^{n_1}]$，$\mathbf{V}=\mathbf{A}_{pos}W^v=[v^1;\cdots;v^{n_1}]$，均为 $\mathbb{R}^{ n_1 \times d_k}$ 中的元素；$W^q$，$W^k$，$W^v$ 均为 $\mathbb{R}^{ n_2 \times d_k}$ 中的元素；
- $\alpha=\mathbf{Q}\mathbf{K}^T/\sqrt{d_k}$，对行向量应用 Softmax 即可得到 $\alpha'$；
- $\mathbf{B}= \alpha'\mathbf{V}\in\mathbb{R}^{n_1 \times d_k}$，所得行向量即为 $b^i$；
- 在上述表述中，计算 $\mathbf{Q}, \mathbf{K}, \mathbf{V}$ 的 $\mathbf{A}_{pos}$ 也可以替换成其他合理的维度为 ($n_1$, $n_2$) 的输入 $\mathbb{Q}, \mathbb{K}, \mathbb{V}$，从而 $\mathbf{Q}=\mathbb{Q}W^q=[q^1;\cdots;q^{n_1}]$，$\mathbf{K}=\mathbb{K}W^k=[k^1;\cdots;k^{n_1}]$，$\mathbf{V}=\mathbb{V}W^v=[v^1;\cdots;v^{n_1}]$；当 $\mathbb{Q}=\mathbb{K}=\mathbb{V}=\mathbf{A}_{pos}$ 时为**自注意力机制 self-attention**。


<br>
<p float="left">
  <img src="./images/trs05.png" alt="Image 8" width="30%"  height="150"/>
  <img src="./images/trs07.png" alt="Image 10" width="30%"  height="150"/>
  <img src="./images/trs08.png" alt="Image 11" width="30%"  height="150"/>
</p>
<br>

**带掩码的自注意力 masked self-attention**（下图一）指在考虑当前位置的词时，仅关注自己和之前词的关系。其出现在 Transformer 的解码器中，目的是为了确保模型按照正确的顺序进行输出。总结成 matrix notations, 记 decoder 当前的输入经过位置编码后为 $\mathbf{A}_{pos}$，有:
- $\mathbf{Q}=\mathbf{A}_{pos}W^q=[q^1;\cdots;q^{n_1}]$，$\mathbf{K}=\mathbf{A}_{pos}W^k=[k^1;\cdots;k^{n_1}]$，$\mathbf{V}=\mathbf{A}_{pos}W^v=[v^1;\cdots;v^{n_1}]$，均为 $\mathbb{R}^{ n_1 \times d_k}$ 中的元素；
- $\alpha=\mathbf{Q}\mathbf{K}^T/\sqrt{d_k} - 10^9\mathbf{M}$，其中 $\mathbf{M}\in\mathbb{R}^{n_1\times n_1}$ s.t. $\mathbf{M}_{i,j}=1_{i< j}$ 为掩码，主对角线以上为 1 其余部分为 0；对行向量应用 Softmax 即可得到 $\alpha'$（被掩码覆盖部分为一个极大的负数，取 Softmax 对应值接近于 0）；$\mathbf{Q}\mathbf{K}^T$ 的维数为 $\mathbf{Q}$ 包含的词数 $\times$ $\mathbf{K}$ 包含的词数；
- $\mathbf{B}= \alpha'\mathbf{V}\in\mathbb{R}^{ n_1 \times d_k}$，所得行向量即为 $b^i$。
<br>

**多头（$n_h$）注意力 multi-head attention** 多头自注意力（下图三）只是将上述的 self-attention Mechanisms 重复 $n_h$ 次，其中的训练参数为 $\{(W^{q,h},W^{k,h},W^{v,h})\}_{h=1}^{n_h}$ 以及一个额外的 $W^B\in\mathbb{R}^{d_k\times n_2}$，其中 $W^{q,h},W^{k,h},W^{v,h}\in\mathbb{R}^{n_2\times(d_k/n_h)}$。在利用 $\{(W^{q,h},W^{k,h},W^{v,h})\}_{h=1}^{n_h}$ 分别得到 $\{\mathbf{B}^{h}\}_{h=1}^{n_h}$ 后堆叠，再进行一次额外的线性变换，得到 $\mathbf{B}=[\mathbf{B}^{1},\cdots,\mathbf{B}^{n_h}]W^B$。当用 $\mathbb{Q}, \mathbb{K}, \mathbb{V}$ 替换 $\mathbf{A}_{pos}$ 时（下图二），即为一般的多头注意力机制（一般 $\mathbb{Q} \neq \mathbb{K} = \mathbb{V}$）。一个例子为 Transformer 的 Decoder 中间部分的交叉注意力（下图四）。
<br>
<br>
<table>
    <tr>
        <td>
          <img src="./images/IM12mask.png" width="550" />
        </td>
        <td>
        </td>
        <td>
          <img src="./images/trs15.png" width="300" /> 
        <td>
        </td>
        <td>
        <img src="./images/IM14.png" width="700" />
    </tr>
</table>
<div align="center">
<img src="./images/IM19.png" width=40% height=40%/>
</div>

<br>
<br>

关于 Batch normalization (下图行一) 和 Layer normalization (下图行二) 的[区别](https://www.pinecone.io/learn/batch-layer-normalization/)：Batch normalization 中 $x=[x_1,\cdots,x_n]$ 代表同一特征在不同样本中的取值；Layer normalization 中 $x=[x_1,\cdots,x_n]$ 代表同一样本在不同特征中的取值。此处 transformer 对词嵌入维度 $n_2$ 进行 Layer normalization。

<br>
<br>
<table>
    <tr>
        <td>
          <img src="./images/IM15.png" width="500" />
        </td>
        <td>
        </td>
        <td>
        <img src="./images/IM16.png" width="300" />
    </tr>
</table>
<br>
<br>
<table>
    <tr>
        <td>
          <img src="./images/IM17.png" width="500" />
        </td>
        <td>
        </td>
        <td>
        <img src="./images/IM18.png" width="300" />
    </tr>
</table>


<a id="sub-03-04-02"></a>

### 3.4.2 注意力计算与因果掩码



In [ ]:
# 构造 Mask
def subsequent_mask(mask_size, device='cpu'):
    # np.ones((1,mask_size,mask_size)) 构造传入 size 的全 1 数组
    # np.triu(A, k=+/-d)) 对数组 A 的最后两个维度构成的矩阵，取主对角线上/下第 d 条副对角线往下所有元素为 0（不包括该副对角线）
    # 1 表示被遮掩的部分
    if isinstance(mask_size, int):
        mask_size = (mask_size, mask_size)
    M = np.triu(np.ones((1,1,*mask_size)), k=1) # 四个维度，前两个维度分别为 batch_size * n_head
    return torch.from_numpy(M).to(device)

# 计算 scaled dot-product attention
def scale_dot_product_attention(Q, K, V, mask=None):
    # Q, K, V 和输出均为四个维度，前两个维度分别为 batch_size * n_head
    d_k = Q.size(-1)
    alpha = torch.matmul(Q, K.transpose(-2,-1)) / sqrt(d_k) # attention score
    if mask is not None:
        alpha = alpha - 1e9*mask.float()
    alpha_p = F.softmax(alpha, dim=-1) # modified attention score (weights)
    return torch.matmul(alpha_p, V), alpha_p

# 将 multi-head attention 的输出进行拼接成三维 tensor
def concat(B):
    return torch.cat([B[:, i, :, :] for i in range(B.size(1))], dim=2)

# 试运行
Q = torch.rand(2,3,5,8).to(device)
K = torch.rand(2,3,5,8).to(device)
V = torch.rand(2,3,5,8).to(device)
mask = subsequent_mask(5,device)
B , alpha_p = scale_dot_product_attention(Q, K, V, mask=mask)

print(mask, end='\n\n')
print(B, end='\n\n')
print(concat(B), end='\n\n')
print(alpha_p, end='\n\n')


<a id="sub-03-04-03"></a>

### 3.4.3 LayerNorm：定义与数值验证



In [ ]:
# 定义 LayerNorm 层
class LayerNorm(nn.Module):

    def __init__(self, embedding_dim, eps=1e-6, device='cpu'):
        super(LayerNorm, self).__init__()
        self.eps = eps
        self.a2 = nn.Parameter(torch.ones(embedding_dim).to(device)) # nn.Parameter(tensor) 将 tensor 转为可训练参数并包含在 model.parameters() 中
        self.b2 = nn.Parameter(torch.zeros(embedding_dim).to(device)) 

    def forward(self, x):
        # x : [batch_size, n1, n2 (embed_dim)]
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        x = self.a2 * (x-mean) / torch.sqrt(var + self.eps) + self.b2                        
        return x

In [ ]:
# 演示 ResLayerNorm 层中 broadcast 规则
x = torch.rand(2,3,5); print(x, end='\n\n')
a2 = torch.arange(5); print(a2, end='\n\n')
b2 = torch.zeros(5); print(b2, end='\n\n')
mean = x.mean(dim=-1, keepdim=True); print(mean, end='\n\n')
var = x.var(dim=-1, keepdim=True, unbiased=False); print(var, end='\n\n')
eps = 1e-6
print((x-mean) / torch.sqrt(var+eps) + b2, end='\n\n')
print(a2 * (x-mean) / torch.sqrt(var+eps) + b2, end='\n\n')


<a id="sub-03-04-04"></a>

### 3.4.4 多头注意力、前馈网络与残差子层



In [ ]:
# 定义 Multi-Head Attention 层
class MultiHeadAttention(nn.Module):

    def __init__(self, embedding_dim, n_head, device='cpu'):
        super(MultiHeadAttention, self).__init__()
        assert embedding_dim % n_head == 0
        self.dv = embedding_dim // n_head 
        self.n_head = n_head
        self.W_q = nn.Linear(embedding_dim, embedding_dim, device=device)
        self.W_k = nn.Linear(embedding_dim, embedding_dim, device=device)
        self.W_v = nn.Linear(embedding_dim, embedding_dim, device=device)
        self.W_B = nn.Linear(embedding_dim, embedding_dim, device=device)

    def forward(self, q, k, v, mask=None):
        # q, k, v : [batch_size, n1, n2 (embed_dim)], self-attention 则 q=k=v=A
        # Project all heads together, then split independent feature slices.
        Q = self.W_q(q).reshape(q.size(0), q.size(1), self.n_head, self.dv).transpose(1, 2)
        K = self.W_k(k).reshape(k.size(0), k.size(1), self.n_head, self.dv).transpose(1, 2)
        V = self.W_v(v).reshape(v.size(0), v.size(1), self.n_head, self.dv).transpose(1, 2)
        B , _ = scale_dot_product_attention(Q, K, V, mask)          # [batch_size, n_h, n_1, n_2/n_h]
        B = self.W_B(concat(B))                                     # [batch_size, n_1, n_2] --> [batch_size, n_1, n_2]
        return B

# 定义 FeedForward 层
class FeedForward(nn.Module):

    def __init__(self, in_dim, out_dim, hidden_dim, n_layer, device='cpu'):
        super(FeedForward, self).__init__()

        self.input_layer = nn.Sequential(nn.Linear(in_dim, hidden_dim, device=device), nn.ReLU())
        self.hidden_layers = nn.Sequential(*[nn.Sequential(nn.Linear(hidden_dim, hidden_dim, device=device), nn.ReLU()) 
                                             for _ in range(n_layer)])
        self.output_layer = nn.Linear(hidden_dim, out_dim, device=device)

    def forward(self, A):
        B = self.input_layer(A)
        B = self.hidden_layers(B)
        B = self.output_layer(B)
        return B

# 定义子层链接结构 (Add & Norm)
class SublayerConnection(nn.Module):

    def __init__(self, embedding_dim, eps=1e-6, device=device):
        super(SublayerConnection, self).__init__()
        self.layer_norm = LayerNorm(embedding_dim, eps, device)

    def forward(self, x, sublayer):
        return self.layer_norm(x + sublayer(x))


<a id="sub-03-04-05"></a>

### 3.4.5 编码器的堆叠



In [ ]:
# 复制 module
def clones(module, N):
    # nn.ModuleList 在作用上等同于常规 list 但会将模型中的参数封装进使用的 model.parameters()中以供调用
    return nn.ModuleList([copy.deepcopy(module) for _ in range(N)]) 

# 构造编码器模块
class EncoderBlock(nn.Module):

    def __init__(self, multihead_attention, feedforward, sublayer_connection):
        super(EncoderBlock, self).__init__()
        self.multihead_attention = multihead_attention
        self.feedforward = feedforward
        self.sublayer_connections = clones(sublayer_connection, 2)

    def forward(self, x):
        x = self.sublayer_connections[0](x, lambda x: self.multihead_attention(x,x,x))
        x = self.sublayer_connections[1](x, self.feedforward)
        return x

# 构造编码器层
class EncoderLayer(nn.Module):
    
    def __init__(self, N, encoder_block):
        super(EncoderLayer, self).__init__()
        self.encoder_blocks = nn.Sequential(*clones(encoder_block, N))
        
    def forward(self, x):
        return self.encoder_blocks(x)


<a id="sub-03-04-06"></a>

### 3.4.6 解码器与交叉注意力



In [ ]:
# 构造解码器模块
class DecoderBlock(nn.Module):

    def __init__(self, masked_multihead_attention_1, masked_multihead_attention_2, feedforward, sublayer_connection):
        super(DecoderBlock, self).__init__()
        self.masked_multihead_attention_1 = masked_multihead_attention_1 # 第一个 attention 要 mask 是为了防止信息泄露
        self.masked_multihead_attention_2 = masked_multihead_attention_2 # 交叉注意力只需屏蔽源序列 padding；不使用目标序列的因果 mask
        self.feedforward = feedforward
        self.sublayer_connections = clones(sublayer_connection, 3)

    def forward(self, x, memory, mask1, mask2):
        # memory 为编码器的输出，作为第二个多头注意力的 key 以及 value
        x = self.sublayer_connections[0](x, lambda x: self.masked_multihead_attention_1(x,x,x,mask1))
        x = self.sublayer_connections[1](x, lambda x: self.masked_multihead_attention_2(x,memory,memory,mask2))
        x = self.sublayer_connections[2](x, self.feedforward)
        return x

# 构造编码器层
class DecoderLayer(nn.Module):
    
    def __init__(self, N, decoder_block):
        super(DecoderLayer, self).__init__()
        self.decoder_blocks = clones(decoder_block, N)
        
    def forward(self, x, memory, mask1, mask2):
        for block in self.decoder_blocks:
            x = block(x, memory, mask1, mask2)
        return x


<a id="sub-03-04-07"></a>

### 3.4.7 输出层、模型整合与前向验证



In [ ]:
# 构造输出层 Generator
class Generator(nn.Module):
    def __init__(self, embedding_dim, num_embeddings, hidden_dim, n_layer, device='cpu'):
        super(Generator, self).__init__()
        self.feedforward = FeedForward(embedding_dim, num_embeddings, hidden_dim, n_layer, device)
        self.soft_max = nn.Softmax(dim=-1)

    def forward(self, x):
        return self.soft_max(self.feedforward(x)) # 每个词向量都对应一个下一个token的概率分布

In [ ]:
# 整合架构
class EncoderDecoder(nn.Module):
    def __init__(self, encoder_layer, decoder_layer, source_embed, target_embed, positional_encoding, generator):
        super(EncoderDecoder, self).__init__()
        self.encoder_layer = encoder_layer
        self.decoder_layer = decoder_layer
        self.source_embed = source_embed
        self.target_embed = target_embed
        self.positional_encoding = positional_encoding
        self.generator = generator

    def forward(self, source, target, mask1, mask2):
        # source <-> x
        # target <-> y
        memory = self.encoder_layer(self.positional_encoding(self.source_embed(source)))
        output = self.decoder_layer(self.positional_encoding(self.target_embed(target)), memory, mask1, mask2)
        output = self.generator(output)
        return output

In [ ]:
# 整合试运行
embedding_dim, n_head = 512, 8
num_embeddings = 10
N = 6

# 通用模块
source_embed = Embeddings(num_embeddings, embedding_dim, device=device)    
target_embed = Embeddings(num_embeddings, embedding_dim, device=device) 
positional_encoding = PositionalEncoding(embedding_dim, device=device) 

# 编码器部分
multihead_attention = MultiHeadAttention(embedding_dim, n_head, device=device) 
feedforward_e = FeedForward(embedding_dim, embedding_dim, embedding_dim, n_layer=1, device=device)
sublayer_connection_e = SublayerConnection(embedding_dim, device=device)
encoder_block = EncoderBlock(multihead_attention, feedforward_e, sublayer_connection_e)
encoder_layer = EncoderLayer(N, encoder_block)

# 解码器部分
masked_multihead_attention_1 = MultiHeadAttention(embedding_dim, n_head, device=device) 
masked_multihead_attention_2 = MultiHeadAttention(embedding_dim, n_head, device=device) 
feedforward_d = FeedForward(embedding_dim, embedding_dim, embedding_dim, n_layer=3, device=device)
sublayer_connection_d = SublayerConnection(embedding_dim, device=device)
decoder_block = DecoderBlock(masked_multihead_attention_1, masked_multihead_attention_2, feedforward_d, sublayer_connection_d)
decoder_layer = DecoderLayer(N, decoder_block)
generator = Generator(embedding_dim, num_embeddings, embedding_dim, n_layer=1, device=device)

# 整合
encoder_decoder = EncoderDecoder(encoder_layer, decoder_layer, source_embed, target_embed, positional_encoding, generator)

# 运行
x = torch.tensor([[1,4,6], [2,1,5]])     # encoder input, 两个 token_id 向量 2*3，代表两句话，每句话 3 个 token；此时 x 在 cpu 上

y1 = torch.tensor([[1], [2]])            # decoder input, 两个 token_id 向量 2*2，代表两句话，每句话 2 个 token；此时 x 在 cpu 上
mask1 = subsequent_mask(mask_size=y1.size(1), device=device)
mask2 = None # Source has no padding; cross-attention can see all source tokens.
output = encoder_decoder(x,y1,mask1,mask2)
print(output, end='\n\n')

y2 = torch.tensor([[1,4], [2,1]])         # decoder input, 两个 token_id 向量 2*2，代表两句话，每句话 2 个 token；此时 x 在 cpu 上
mask1 = subsequent_mask(mask_size=y2.size(1), device=device)
mask2 = None
output = encoder_decoder(x,y2,mask1,mask2)
print(output, end='\n\n')

y3 = torch.tensor([[1,4,6], [2,1,5]])     # decoder input, 两个 token_id 向量 2*2，代表两句话，每句话 2 个 token；此时 x 在 cpu 上
mask1 = subsequent_mask(mask_size=y3.size(1), device=device)
mask2 = None
output = encoder_decoder(x,y3,mask1,mask2)
print(output, end='\n\n')

一个重要的观察是，在固定权重下，Decoder 的输入每新增一个 token，并不会改变前一步输出的预测分布（下一步预测不会改变前一步预测）。

---

<a id="part-03-05"></a>

## 3.5 训练目标与 Teacher Forcing

> **本节目标**：区分输入右移、目标 token、logits 与概率，理解 Teacher Forcing。

**小节导航**

- [3.5.1 Teacher Forcing 与训练目标](#sub-03-05-01)
- [3.5.2 Logits、概率与交叉熵输入](#sub-03-05-02)

[返回章节目录](#chapter-03)



<a id="sub-03-05-01"></a>

### 3.5.1 Teacher Forcing 与训练目标


采用交叉熵损失，以 batch_size = 1 的情况为例：
- 对 label 中的每个 token 进行 token_id 编码，得到 token_id 矢量 $\mathbf{y}\in\mathbb{R}^{1\times n_1}$；
- 在训练时，将 $[\mathrm{begin}, \mathbf{y}]$ 作为 Decoder 的输入一次性传入（Teacher Forcing），得到对应的预测 distribution，大小为 $(n_1+1)\times N_w$（包含 end 符号），记作 $\mathbf{Z}$；
- 对 $\mathbf{y} := [\mathbf{y}, \mathrm{end}]$ 和 $\mathbf{Z}$ 计算交叉熵损失 CEL
$$CEL(\mathbf{y},\mathbf{Z}) = -\sum_{i=1}^{n_1+1}\log(\mathbf{Z}[i,\mathbf{y}[i]])$$
- 若 batch_size = bs != 1，则有若干个 $\mathbf{Y} = [\mathbf{y}^1;\cdots;\mathbf{y}^{bs}]$ 和 $\mathbf{Z} = [\mathbf{Z}^1;\cdots;\mathbf{Z}^{bs}]$,
$$CEL(\mathbf{Y},\mathbf{Z}) = -\sum_{k=1}^{bs}\sum_{i=1}^{n_1+1}\log(\mathbf{Z}^k[i,\mathbf{y}^k[i]])$$
或者
$$MCEL(\mathbf{Y},\mathbf{Z}) = -\frac{1}{bs}\sum_{k=1}^{bs}\sum_{i=1}^{n_1+1}\log(\mathbf{Z}^k[i,\mathbf{y}^k[i]])$$


<a id="sub-03-05-02"></a>

### 3.5.2 Logits、概率与交叉熵输入



PyTorch 的 CrossEntropyLoss 接收原始 logits，内部等价于 LogSoftmax + NLLLoss。本笔记的 Generator 为展示而输出概率；实际训练若使用 CrossEntropyLoss，应输入其 `feedforward(x)` 的 logits，而不是 Generator 的 Softmax 输出。上式按序列求和再按 batch 平均；PyTorch 默认 mean 则按有效 token 平均（如有 padding 应配合 ignore_index）。
<div align="center">
<img src="./images/IM20.png" width=40% height=40%/>
</div>

---

<a id="part-03-06"></a>

## 3.6 训练归一化与稳定性

> **本节目标**：连接梯度传播、残差和归一化策略，再进入 ViT 归一化实践。

**小节导航**

- [3.6.1 Bottom-to-top (B2T) Connection](#sub-03-06-01)
- [3.6.2 DeepNorm](#sub-03-06-02)
- [3.6.3 Spike No More](#sub-03-06-03)
- [3.6.4 Mix-LN](#sub-03-06-04)
- [3.6.5 实践：ViT 的归一化顺序](#sub-03-06-05)

[返回章节目录](#chapter-03)


Training Normalization & Stabilization

<a id="sub-03-06-01"></a>

### 3.6.1 [Bottom-to-top (B2T) Connection](https://arxiv.org/pdf/2206.00330)

- Post-LN with deep Trans-formers (ten or more layers): the training is often **unstable**.
- Post-LN with shallow Transformers (six or fewer layers): consistently achieve **better performance** than Pre-LN.
- Post-LN can achieve better performance if training succeeds. In Pre-LN, a deeper layer has a smaller gradient norm, implying that **higher layers in Pre-LN are not sufficiently effective**.
- This study first investigates the reason for these discrepant observations empirically and theoretically and made the following discoveries:
    - 1, the LN in Post-LN is the **main source of the vanishing gradient** problem that leads to unstable training, whereas Pre-LN prevents it;
    - 2, Post-LN tends to **preserve larger gradient norms** in higher layers during the back-propagation, which may lead to effective training.
- We propose a method that can provide both high stability and effective training by a simple modification of Post-LN [(Code)](https://github.com/takase/b2t_connection).<br>
<div align="center">
    <img src="./images/IM57.png" width=30% height=30%/>
</div>
<br>

**Theoretical Explanation**
-  Let x be an input of a sublayer, and $\mathcal{F}(\cdot)$ be a sublayer of a Transformer, such as a feed-forward network or multi-head attention. Post-LN and Pre-LN are defined as follows: $$\mathrm{PostLN}(x)=\mathrm{LN}(x+\mathcal{F}(x)),$$ $$\mathrm{PreLN}(x)=x+\mathcal{F}(\mathrm{LN}(x)).$$
- Taking the derivatives $$\partial_x\mathrm{PostLN}(x)=\partial_{x+\mathcal{F}(x)}\mathrm{LN}(x+\mathcal{F}(x))\cdot(I + \partial_x\mathcal{F}(x)),$$ $$\partial_x\mathrm{PreLN}(x)=I+\partial_{\mathrm{LN}(x)}\mathcal{F}(\mathrm{LN}(x))\cdot\partial_x\mathrm{LN}(x).$$
- [The magnitude of gradient through LN is inversely proportional to the magnitude of its input](https://arxiv.org/pdf/2002.04745): $\parallel\partial_{x}\mathrm{LN}(x)\parallel=\mathcal{O}(\sqrt{d}/\parallel x \parallel)$. In Post-LN, when **the magnitude of the input $x$ is significantly larger than the square root of the dimension**, which is usually the case when [neither warm-up nor good initialization is given](https://arxiv.org/pdf/2203.00555), the training will be unstable due to the vanishing gradient.
- In Pre-LN, the derivative of the residual connection $I$ is isolated from the term related to the derivative of the layer normalization, implying that **the residual connection in Pre-LN prevents the vanishing gradient** because it retains the gradients of upper layers even if the derivative of the layer normalization decreases gradients drastically. Also, in Pre-LN, an input $x$ skips over the sub-layer by the residual connection. Thus, the input x is directly connected to the final layer output, resulting in **high similarities** between the outputs of the various layers.
<br>
<br>

**B2T Connection**
- Propose a residual connection that skips over all layer normalization except the final one in each layer for Post-LN blocks to prevent the vanishing gradient.
  $$\mathrm{BlockOutput}(x)=\mathrm{LN}(x+x_{ffn}(x)+\mathrm{FFN}(x_{ffn})).$$ In an encoder-side, $x_{ffn}(x)=\mathrm{LN}(\mathrm{SelfAttn}(x)+x).$ Taking the derivative w.r.t. $\mathrm{B2T}(x)=x+x_{ffn}(x)+\mathrm{FFN}(x_{ffn})$ , we have
  $$\partial_x\mathrm{B2T}(x)=I+(I+\partial_{x_{ffn}}\mathrm{FFN}(x_{ffn}))\cdot \partial_xx_{ffn}(x).$$
  Thus, the term $x$ helps to preserve the gradient.
<div align="center">
    <img src="./images/IM58.png" width=20% height=20%/>
</div>
<br>
<br>

**Possible Variant**
- Add weights for B2T connection to further mitigate the decreasing gradients in back-propagation. Let $\alpha=\min\{N/12,N^{-0.15}\}$ and $\beta=d^{-0.2}$ ($N$ is the number of layers and $d$ is the dimension of the input vectors $x$), and $$\mathrm{BlockOutput}(x)=\alpha x+\beta(x_{ffn}(x)+\mathrm{FFN}(x_{ffn})).$$
- Possibly setting $\alpha$ and $\beta$ as learnable parameters?


<a id="sub-03-06-02"></a>

### 3.6.2 [DeepNorm](https://arxiv.org/pdf/2203.00555)

- The proposed method combines the good performance of Post-LN and the stable training of Pre-LN.
- The proposed can stabilize a Transformer model with up to 1,000 layers.
- The analysis begins with the observation: **better initialization methods** stabilize the training of Transformer. 

**DeepNorm**
- $\mathrm{DeepNorm}(x)=\mathrm{LN}(\alpha x + \mathcal{F}(x))$ compared to $\mathrm{PostLN}(x)=\mathrm{LN}(x+\mathcal{F}(x)).$ Also, down-scale the parameters during initialization while applying more proper warm-up and initialization methods ([**Xavier initialization**](https://proceedings.mlr.press/v9/glorot10a/glorot10a.pdf)).
<br>
<br>
<div align="center">
    <img src="./images/IM59.png" width=60% height=60%/>
    <img src="./images/IM62.png" width=50% height=50%/>
</div>
<br>
<br>

- The instability with Post-LN starts from the large model update, caused by **poor warm-up and initialization** at the beginning of training, rendering the model trapped in a bad local optimum, which in turn **increases the magnitude of the model update and inputs to each LN (observation in figure below)** and leads to **the problem of the vanishing gradient** during training.  The vanishing gradients make it difficult to escape from the local optimum, further destabilising the optimization.
<br>
<br>
<div align="center">
    <img src="./images/IM61.png" width=60% height=60%/>
</div>
<br>
<br>


- The article proves that, under certain circumstances, for an encoder-only Transformer $\mathrm{DeepNet}(x,\Theta)$, the model update $\parallel\Delta \mathrm{DeepNet}(x,\Theta)\parallel:=\parallel \mathrm{DeepNet}(x,\Theta+\Delta\Theta)-\mathrm{DeepNet}(x,\Theta)\parallel$ satisfies $$\parallel\Delta \mathrm{DeepNet}(x,\Theta)\parallel=\mathcal{O}\left(\sum_{\theta\in\Theta}\frac{\parallel\theta\parallel}{\alpha}\parallel\Delta\theta\parallel\right).$$ For a more complicate encoder-decoder structure, see [article](https://arxiv.org/pdf/2203.00555). By bounding $\parallel\Delta \mathrm{DeepNet}(x,\Theta)\parallel$ using the above theorem, we bound the inputs of $\mathrm{LN}(x)$ (by observation) and $\partial_x\mathrm{PostLN}(x)$, thus mitigating the problem of the vanishing gradient with Post-LN. Figure below shows that the model update of DEEPNET is nearly constant, while the model update of Post-LN is exploding. 
<div align="center">
    <img src="./images/IM60.png" width=60% height=60%/>
</div>


<a id="sub-03-06-03"></a>

### 3.6.3 [Spike No More](https://arxiv.org/abs/2312.16903)

- The analysis was based on the **observation** that **when the gradient norms grow suddenly during LLM pre-training, the loss spike problem is likely to occur**.
- **Assume that we can prevent the loss spike problem by maintaining the gradient norm small.**
- Define the gradient norms here as $\parallel \partial_x\mathcal{L} \parallel_2$ where $x=X[:,j]$, $X\in\mathbb{R}^{d\times L}$, $d$ is the input dimension and $L$ is the sequence length, i.e., the gradient of loss w.r.t. a single token vector of the model input.
- To suppress the upper bound of gradient norms, the author provides two conditions: **small sub-layers sds (standard deviations)** and **large shortcut (below $x$ and $x'$ for each layer) sds**. 
- Theoretical contribution: if, for each layer, $x'=x+\mathrm{Attn}(\mathrm{LN}(x))$ and $y=x'+\mathrm{FNN}(\mathrm{LN}(x'))$ with $$\mathrm{Attn}(x)=W_O(\mathrm{concat}(\mathrm{head}_i(x))),$$ $$\mathrm{FNN}(x)=W_2(\mathrm{Activate}(W_1x)),$$ and all elements of $x,x',W_O,W_1,W_2$ are i.i.d. random variables with mean 0 and sd $\sigma_x,\sigma_{x'},\sigma_O,\sigma_1,\sigma_2$ (valid if we initialize parameters with the normal distribution), we need $\sigma_O\ll\sigma_x$ and $\sigma_1\sigma_2\ll\sigma_{x'}$ across all layers to ensure a small upper bound of the gradient norm.
- Small $\sigma_O,\sigma_1,\sigma_2$ can be obtained via initialization $W\sim\mathcal{N}(0,\sigma^2=\frac{2}{5d})$ where d is the model dimension.
- Large $\sigma_x,\sigma_{x'}$ can be obtained via **Scaled Embed** and **Embed LN**. **The Scaled Embed** scales embeddings
with an appropriate value, for example, multiplying embeddings by $\sqrt{d}$; **The Embed LN** applies the LN to the embedding layer.
<br>
<br>
<div align="center">
    <img src="./images/IM63.png" width=65% height=65%/>
</div>
<br>
<br>
<div align="center">
    <img src="./images/IM64.png" width=50% height=50%/>
</div>
<br>
<br>
<div align="center">
    <img src="./images/IM65.png" width=50% height=50%/>
</div>
<br>
<br>
-  The larger the learning rate we used, the more frequent the spikes occurred in **Vanilla**. In contrast, Scaled Embed stabilized the training.
<br>
<br>
<div align="center">
    <img src="./images/IM66.png" width=60% height=60%/>
</div>


<a id="sub-03-06-04"></a>

### 3.6.4 [Mix-LN](https://openreview.net/pdf?id=BChpQU64RG)

- Applies Post-LN to the earlier layers (25% layers) and Pre-LN to the deeper layers (75%), ensuring more uniform gradient norms across layers and improving final model performance. This allows all parts of the network — both shallow and deep layers — to contribute effectively to training. 


<a id="sub-03-06-05"></a>

### 3.6.5 实践：ViT 的归一化顺序


在图像分类中验证本节的残差与归一化概念，进入 [ViT 实践](practices/vit-normalization/README.md)。先运行下面的微型前向/反向检查，再准备 ImageNet100 比较曲线。

实现解析：patch embedding 将空间块转换为 token；`Block.forward` 控制 LN 在子层前还是残差和之后；`MixTypeLNTransformer` 控制两种块的深度分配。`run_epoch` 通过 model.train(False) 关闭验证 dropout，按样本数累计指标。先观察输出形状及梯度，再比较不同 alpha，不能只对比一次随机前向输出就推断性能。


In [ ]:
import sys
from pathlib import Path
practice = Path('practices/vit-normalization').resolve()
sys.path.insert(0, str(practice))
from train import make_model
import torch
tiny = make_model('PostPre', alpha=0.5, image_size=32, patch_size=8,
                  num_classes=3, dim=32, depth=2, heads=2, dim_head=16, mlp_dim=64)
logits = tiny(torch.randn(2, 3, 32, 32))
loss = torch.nn.functional.cross_entropy(logits, torch.tensor([0, 2]))
loss.backward()
print(logits.shape, float(loss.detach()))


---

<a id="part-03-07"></a>

## 3.7 参数与内存高效微调

> **本节目标**：分别理解减少可训练参数与降低训练内存开销的方法。

**小节导航**

- [3.7.1 参数高效微调 Parameter-Efficient Fine-Tuning PEFT](#sub-03-07-01)
- [3.7.2 内存高效微调 Memory-Efficient Fine-tuning MEFT](#sub-03-07-02)

[返回章节目录](#chapter-03)


模型微调 Tuning

<a id="sub-03-07-01"></a>

### 3.7.1 参数高效微调 Parameter-Efficient Fine-Tuning PEFT


<a id="sub-03-07-01-01"></a>

#### 3.7.1.1 概述

- What is Fine-Tuning : 调整已完成预训练的大模型，使其适配不同的下游任务。
- Why PEFT : 全参数微调 (Full Fine-tuning) 需要消耗大量的计算资源，不能在较小的设备上完成训练。PEFT 参数高效微调方法是目前大模型在工业界应用的主流方式之一。PEFT方法仅微调少量或额外的模型参数，固定大部分预训练参数，大大降低了计算和存储成本，同时最先进的PEFT技术也能实现了与全量微调相当的性能。以下是几种常见的传统 PEFT 方法：
    - BitFit : 下图一， 仅微调（线性层的）偏置项 bias；大模型表现较差。
    - Adapter : 下图二（适配层），在架构中添加额外的MLP层进行训练；便于不同任务的添加（加不同的 Adapter 层即可），但会在一定程度上增大模型推理阶段的延迟。
    - (Soft) Prompt Tuning : 下图三（提示词微调），不改变模型的基本架构和参数，仅在词嵌入阶段为不同的任务添加可学习的嵌入向量作为提示（在推理阶段，提示词在一开始就固定，不会随着模型的推理进程发生变化，即 discrete prompt），以引导模型输出对特定任务的响应。缺点是由于增大了输入的长度，会在一定程度上增大模型推理阶段的延迟且占用下游任务实际可使用的序列长度。
    - [Prefix Tuning](https://lightning.ai/pages/community/article/understanding-llama-adapters/) : 下图四（前缀微调），在每个 Transformer 层都加入一个可训练的 MLP 层作为不同任务嵌入张量的前缀（在推理阶段，提示词随着模型的推理进程发生变化，即 continuous prompt）。[该方法同样会增大输入长度，导致推理延迟，占用下游任务实际可使用的序列长度；且训练效果不一定随这参数数量的增加而增强](https://arxiv.org/pdf/2106.09685)。


<br>
<div align="center">
<img src="./images/IM25.png" width=20% height=20%/>
<img src="./images/IM26.png" width=35% height=50%/>
<img src="./images/IM27.png" width=23% height=20%/>
<img src="./images/IM28.png" width=40% height=20%/>
</div>


<a id="sub-03-07-01-02"></a>

#### 3.7.1.2 Low-Rank Adaptation: LoRA [低秩适配](https://arxiv.org/pdf/2106.09685)

LoRA 及其衍生变体是当前实现大模型 PEFT 的主流方法。其目的是为了实现在不破坏模型的原始性能的情况下，进一步加速预训练模型的微调过程。较之上文的传统方法，有如下的核心优势：**消除了模型推理阶段的延迟（no additional inference latency）；不破坏模型原始性能；可以通过替换少量参数矩阵快速适应新任务，提高任务切换效率；优化了模型的计算资源和存储空间**。其核心原理及假设是**微调时参数的更新主要集中在少量参数上（主要集中在低维子空间中）**。风险是**若参数更新实际发生在高维子空间中，则可能导致重要信息的遗漏**，且不能在不额外增加延迟的基础上一次性输出多项不同任务的结果。
<div align="center">
<img src="./images/IM21LoRA.png" width=20% height=20%/>
</div>
<br>

记 $\mathbf{W}_{pretrained}$ 为预训练参数矩阵（在 Transformer 中可以是 $W^q,\ W^k,\ W^v,\ W^B/W^O$ 或者线性层参数，或者 embedding 层），$\mathbf{W}_{finetuned}$ 为微调后的参数矩阵。微调的目的是训练一个 $\Delta\mathbf{W}=\mathbf{A}\mathbf{B}$ 从而使得 $\mathbf{W}_{finetuned}=\mathbf{W}_{pretrained}+\Delta\mathbf{W}$。假设 $\mathbf{W}_{pretrained}\in \mathbb{R}^{{d_1}\times{d_2}}$，则 $\mathbf{A}\in \mathbb{R}^{{d_1}\times{r}}$，$\mathbf{B}\in \mathbb{R}^{{r}\times{d_2}}$，且 $r\ll\min\{d_1,d_2\}$。此时模型参数的训练量从 $\mathcal{O}(d_1d_2)$ 减少为 $\mathcal{O}(rd_2)$。初始化时，选择 $\mathbf{A}\sim\mathcal{N}(0,1)$，$\mathbf{B}=\mathbf{O}$。选择 $\mathbf{A}\sim\mathcal{N}(0,1)$ 的目的是为了确保初始梯度的有效传播，避免梯度消失或爆炸；同时为模型提供足够的随机性。$\mathbf{B}$ 初始化为全 0 矩阵的目的是确保微调后的参数不会与训练参数偏离过远。

<br>
<div align="center">
<img src="./images/IM22.png" width=50% height=50%/>
</div>
<br>

上图为 LoRA 和其他 fine-tuning 方法的比较；下二图为在 Transformer 不同位置应用不同低秩适配的效果比较。实验表明，微调效果并不是 $r$ 越大越好，且与应用 LoRA 的位置有很大关系。

<br>
<div align="center">
<img src="./images/IM23.png" width=50% height=50%/>
</div>
<br>
<div align="center">
<img src="./images/IM24.png" width=50% height=50%/>
</div>
<br>

LoRA 的变体：QLoRA，在 LoRA 的基础上引入量化机制（见下文）+ Paged Optimization；LongLoRA 用于训练长文本 LLM；[ReLoRA](https://arxiv.org/pdf/2307.05695) 周期性使用 $\mathbf{W}_{pretrained}+\Delta\mathbf{W}$ 代替原来的 $\mathbf{W}_{pretrained}$。进一步总结 LoRA 高效的原因：
- 消除了模型推理阶段的延迟；
- 不破坏模型原始性能；
- 可以通过替换少量参数矩阵快速适应新任务，提高任务切换效率；
- 实验表明，$\mathbf{A},\mathbf{B}$ 仅在 top singular-vector directions 张成的空间重合度较高（下图一二），因此采用低秩更新就足够达到比较好的效果，进一步优化了模型的计算资源和存储空间；
- 实验表明，LoRA 有潜力放大在预训练模型中已经学到但未被强调的特定下游任务的重要特征（下图三）；
- [相关理论基础见此处](https://zhuanlan.zhihu.com/p/719930225)。

<br>
<div align="center">
<img src="./images/IM29.png" width=60% height=60%/>
<img src="./images/IM30.png" width=60% height=60%/>
</div>
<br>
<div align="center">
<img src="./images/IM31.png" width=60% height=60%/>
</div>
<br>


<a id="sub-03-07-02"></a>

### 3.7.2 内存高效微调 Memory-Efficient Fine-tuning MEFT


<a id="sub-03-07-02-01"></a>

#### 3.7.2.1 Gradient Low-Rank Projection: GaLore [梯度低秩投影](https://arxiv.org/pdf/2403.03507)

上述 LoRA 通常只适用于已完成预训练的模型，而不能直接用于模型预训练（因为最优权重矩阵一般不是低秩的）。尽管 ReLoRA 可以在一定程度上实现将 LoRA 应用于预训练（过程中参数矩阵的秩会增加），但效果往往欠佳，除非在预训练的初期先对模型进行全参数预热。LoRA 及其变体在预训练上的缺陷可能源于两方面：1)  the optimal weight matrices may not be low-rank；2) reparameterization changes the gradient training dynamics。

GaLore 梯度低秩投影是一种能够大大降低内存占用的**全参微调**或**全参数预训练**技术，且首次实现了在消费级显卡上全参数预训练大模型的操作。其核心思想是**将参数矩阵映射到低维子空间上进行梯度更新后再映射回原空间（从而大大降低了优化器参数和权重梯度所占用的内存）i.e.，subspace learning**。见下图一二，下图二中的 $\rho_t(\cdot)$ 可视为所使用的优化器（实际上是 entry-wise stateful [gradient regularizer](https://arxiv.org/pdf/2406.09723)）。文章建议对每一步的梯度进行 (compact) SVD 分解 $G_t=U_tS_tV_t^T$ 并取 $P_t=U_t\in\mathbb{R}^{m\times r},\ Q_t=V_t\in\mathbb{R}^{n\times r}$，其中超参数为 compact SVD 的秩 $r$。实际应用中可能只会单独应用 $P_t$ 或者 $Q_t$ 中的一个进行投影。即，将下图二中的 $\rho_t(P_t^TG_tQ_t)$ 替换成 $\rho_t(P_t^TG_t)$ 或者 $\rho_t(G_tQ_t)$（下图三）。此外，还有一个比较重要的超参数是更新 $P_t,\ Q_t$ 的频率 $T$（步更新一次）。
<div align="center">
<img src="./images/IM32.png" width=30% height=40%/>
    <img src="./images/IM35.png" width=30% height=40%/>
    <img src="./images/IM36.png" width=30% height=40%/>
</div>
<br>

该技术的核心原理基于（文章所证明的）**特定网络结构（reversible models）的参数矩阵的更新梯度会随着训练逐渐变成低秩矩阵**。下图一：不等式右侧第二项随 t 收敛于 0，$G_{t_0}^{\parallel}=\mathrm{vec}^{-1}(g_{t_0}^{\parallel})$，其中 $g_{t_0}^{\parallel}$ 是$g_{t_0}=\mathrm{vec}(G_{t_0})$ 在 $S$ 对应特征值 $\lambda_1$ 下特征空间内的投影。$\mathrm{sr}(G)=\frac{\parallel G\parallel^2_F}{\parallel G\parallel^2_2}$ 是对 $\mathrm{rank}(G)$ 的近似。PSD 表示半正定。作者在下图二中为 $\mathrm{sr}(G_{t_0}^{\parallel})$ 给出了估计 $\mathrm{sr}(G_{t_0}^{\parallel})<n-N'$，其中 $N'=\mathrm{rank}(\{f_i\}_{i=1}^N)\le n$，$f_i\in\mathbb{R}^{n}$ 是每层嵌套的输出（下列推论使用了放缩 $\min\{a-b,b\}\le a/2$）。
<br>
<br>
<table>
    <tr>
        <td>
          <img src="./images/IM33.png" width="300" />
        </td>
        <td>
            <img src="./images/IM34.png" width="400" />
        </td>
    </tr>
</table>

实验结果表明
- GaLore 对所应用的优化器以及超参数秩 $r$ 的选择比较稳定（下图一 Fig3）；
- 频繁更新子空间或者频繁不更新子空间（$T$ 过大或过小）都会影响训练效果，对于低秩（小 $r$）影响更明显（下图二 Fig5，左）；
- 固定 $T$，在一定范围内再增大 $r$ 不会再对模型训练效果产生显著的影响（下图二 Fig5，右）；
<br>
<div align="center">
<img src="./images/IM37.png" width=70% height=70%/>
<img src="./images/IM38.png" width=40% height=50%/>
</div>
<br>


<a id="sub-03-07-02-02"></a>

#### 3.7.2.2 Layerwise Importance Sampled AdamW: [LISA](https://arxiv.org/pdf/2403.17919)

与 GaLore 一样，LISA 也是一种内存高效微调方法（不能用于直接预训练，但可以和 ReLoRA 一样用于持续预训练，i.e., 需要预热）；也可以归类于 PEFT 中的 selective methods。作者研究了LoRA在微调任务中的层间特性，并观察到**不同层之间的权重范数存在一致的偏斜现象**。利用这一结果，在**冻结大部分预训练中间层参数后，再随机采样解冻部分层参数进行训练**。实验结果显示，在类似或更低的GPU内存消耗下，LISA在下游微调任务中超越了LoRA 甚至 GaLore 和全参数微调。
<br>
<div align="center">
<img src="./images/IM39.png" width=30% height=30%/>
</div>
<br>
作者首先记录了在采用 LoRA 进行参数微调过程中，各层权重的平均范数，发现嵌入层或头部层在 LoRA 中的权重范数显著大于中间层。然而，在全参数训练设置下，这种现象并不明显（下图）。此处的第 $l$ 层权重 $\theta^l$ 在训练过程 $[0,T]$ 中的的平均范数定义为 $$\mathbf{w}^l := \frac{1}{T}\sum^T_{t=1}\parallel\theta_t^l\parallel_F$$
<br>
<div align="center">
<img src="./images/IM40.png" width=70% height=70%/>
</div>
<br>

算法实现：首先，保持模型的 embedding 和尾层参数一直解冻（全程可更新）；接着，对剩下每一层制定一个采样分布。假设排除首尾层还有 $N_{L}$ 层参数，则该分布通常为一个均匀分布，记为 $p_l=\frac{1}{N_L}$（原文中 $p_l=\frac{\gamma}{N_L}$ 为该层被采样的概率）。给定每次采样的层数 $\gamma$ 以及重采样周期 $K$ 并进行如下循环：
- 固定当前除了首尾层外的所有参数；
- 根据给定分布均匀 $p_l=\frac{1}{N_L}$ 从被冻结的中间层中选取 $\gamma$ 层解冻以参与本轮参数更新；
- 对当前可训练参数使用 AdamW 优化器迭代 $K$ 次，训练期间学习率可以动态调整；
- 返回第一步并重复。
<br>
<div align="center">
<img src="./images/IM41.png" width=70% height=70%/>
</div>
<br>


<a id="sub-03-07-02-03"></a>

#### 3.7.2.3 Outlier-weighed Layerwise Sampled Low-Rank Projection: [OwLore](https://github.com/pixeli99/OwLore?tab=readme-ov-file)

OwLore 可以视为对 LISA 采样步骤的进一步改进，同时在应用 LISA 的过程中进一步结合了 GaLore。文章主要指明并改进了 LISA 的两种缺陷：
- LISA 对中间层采用均匀采样，可能导致次优性能；
- LISA 对被选中的采样层以全秩方式进行微调，随着采样层数量的增加，这会导致显著的内存增加。
这在 OwLore 中分别对应了两种改进策略：
- 根据 layerwise outlier distribution 对模型各层进行采样以更新参数；
- 对被选中的采样层采用 GaLore 进行微调，以进一步优化内存，增大可以被调整的层数。
<br>
<br>
<div align="center">
<img src="./images/IM42.png" width=60% height=60%/>
</div>
<br>

原理：
- 根据 HT-SR 理论，深层神经网络各层参数矩阵的特征值会随着训练的深入偏向重尾分布；
- 所以，在训练模型时，应当更加关注那些特征值服从重尾分布的层所对应的参数矩阵；
- 参数矩阵特征值的重尾程度可以用 PL_Alpha_Hill metric $\mathrm{PL\_Alpha\_Hill}_l$ 度量；
- 实验表明，该度量与文中定义的 outlier metric $D_l$ 显著（正）相关；
- 因此，可以使用各层的 outlier metric 导出的 Layerwise Outlier Distribution (LOD) 构建采样分布。
<br>
<br>
<div align="center">
<img src="./images/IM43.png" width=70% height=70%/>
</div>
<br>

两个度量的定义如下
<br>
<br>
<div align="center">
<img src="./images/IM45.png" width=70% height=70%/>
</div>
<br>
其中 $\mathbf{A}_{ij}$ 的大小反应了前层输入的第 j 个特征对于下层第 i 个输出特征的影响程度。$\mathbf{X}_{j}$ 表示第 $j$ 列，其范数代表了 batch 中所有样本的每个词向量在前一层第 j 个特征的总大小。（2）式中括号内的式子表面判断该层有多少权重对应的 Outlier score 超出了该层的平均水平；分母代表该层涉及的总参量。总体算法的伪代码如下所示
<br>
<br>
<div align="center">
<img src="./images/IM46.png" width=70% height=70%/>
</div>
<br>
训练效果
<br>
<br>
<div align="center">
<img src="./images/IM47.png" width=70% height=70%/>
<img src="./images/IM48.png" width=70% height=70%/>
</div>
<br>
有没有尝试结合 LISA 和 GaLore ?

---

<a id="part-03-08"></a>

## 3.8 模型压缩与剪枝

> **本节目标**：区分压缩、剪枝与量化，衔接 AWQ 校准练习。

**小节导航**

- [3.8.1 常见的压缩手段](#sub-03-08-01)
- [3.8.2 剪枝 Pruning](#sub-03-08-02)
- [3.8.3 实践：从压缩公式到 AWQ 校准](#sub-03-08-03)

[返回章节目录](#chapter-03)


模型压缩 Compression
模型压缩的意义：通过压缩，模型文件会变小，其使用的硬盘空间也会变小，加载到内存或者显存时使用的缓存空间也会变小，并且模型的运行速度还可能会有一些提高。通过压缩，使用模型将消耗更少的计算资源，这可以极大的扩展模型的应用场景，特别是对模型大小和计算效率比较关注的地方，比如手机、嵌入式设备等。


<a id="sub-03-08-01"></a>

### 3.8.1 常见的压缩手段

- 量化 Quantization：量化就是**降低模型参数的数值精度**，比如用 16 位浮点数代替最开始训练出 32 位的浮点数权重。量化也能使计算更快。沿着这个思路，人们继续压缩出了8位、4位、2位的模型，体积更小，使用的计算资源更少。量化技术有很多不同的策略和技术细节，比如如动态量化、静态量化、对称量化、非对称量化等，对于大语言模型，通常采用静态量化的策略，在模型训练完成后，我们就对参数进行一次量化，模型运行时不再需要进行量化计算，这样可以方便地分发和部署。
<br>
<br>
<div align="center">
<img src="./images/IM49.png" width=60% height=60%/>
</div>
<br>
- 蒸馏 Distillation：蒸馏就是把**大模型学习到的行为复制到一个小模型中**。被复制的模型称为教师模型，一般是参数量较大、性能很强的优秀模型，新模型称为学生模型，一般是参数比较少的小模型。蒸馏时，教师模型会根据输入生成多个可能的输出，然后学生模型直接学习这个输入和输出。蒸馏可能会丢失一些信息；另外学生模型可能过度依赖教师模型，导致模型的**泛化能力不佳**。为了让学生模型的学习效果更好，我们可以采用一些方法和策略：1）**引入温度参数**：让模型输出更加平滑的概率分布，方便学生模型捕捉和学习教师模型的输出细节；2）**调整教师模型和学生模型的结构**。
<br>
<br>
<div align="center">
<img src="./images/IM50.png" width=60% height=60%/>
</div>
<br>

- 剪枝 Pruning：剪枝就是**去掉模型中不重要的或者很少会用到的权重**，这些权重的数值一般都接近于0。剪枝的一种实现是**对低于某个阈值的权重进行置 0 （下图二，蓝色为输入矩阵）**。剪枝还会增强模型的可解释性，但对于一些稀疏模型（大部份参数都为0或者接近于0），剪枝所能起到的作用十分有限；对于一些参数比较少的小型模型，剪枝可能导致模型性能的明显下降；对于一些高精度的任务或者应用，也不适合对模型进行剪枝，比如医疗诊断。在实际运用剪枝技术时，通常需要综合考虑剪枝对模型运行速度的提升和对模型性能的负面影响，采取一些策略，比如给模型中的每个参数打分以评估参数对模型性能的贡献。
<br>
<br>
<div align="center">
<img src="./images/IM51.png" width=60% height=60%/>
    <img src="./images/IM52.png" width=60% height=60%/>
</div>
<br>



<a id="sub-03-08-02"></a>

### 3.8.2 剪枝 Pruning


<a id="sub-03-08-02-01"></a>

#### 3.8.2.1 [SparseGPT](https://arxiv.org/pdf/2301.00774): Massive Language Models Can be Accurately Pruned in One-Shot

一般来说模型经过剪枝之后还要再进行一次重新训练，这是十分消耗资源的。本文作者提出了一种新的技术，可以**最大程度地缩短甚至消除剪枝模型的第二次训练时长**。在后面的笔记中我们默认输入矩阵 $X$ 的大小为 $(\mathrm{batch\_size\times seq\_length},C_{in})$ 权重矩阵为 $W\in\mathbb{R}^{C_{in}\times C_{out}}$ 他们之间的线性作用为 $Y=XW$（$W$ 的第 j 列代表第 j 个输出特征的权重，原文采用的是 $Y=WX$）。在开始之前，先介绍一些剪枝问题的核心概念：
- Mask：是一个和 $W$ 大小相同的矩阵 $M$ 其中编码了被**保留**的权重位置，我们记这些位置上的值为 1，剪枝后剩下的权重记为 $W_M$ 或者 $M(W)$。
- Weight Reconstruction：根据 mask 对给定的权重置零后，需要对剩下的权重进行重新调整，这个过程称为权重重构 weight reconstruction。
- Post-Training Pruning：指的是根据已经训练好的模型参数 $\Theta$ 以及 calibration data 计算合适压缩参数 $\hat{\Theta}$ 的过程。
- Layer-Wise Pruning：指的是在 Post-Training Pruning 过程中，逐层对参数进行剪枝和权重重构的过程，对于每一层的输入矩阵 $X_l$ 和原始权重 $W_l$，Layer-Wise Pruning 旨在解决子问题 $$\mathrm{argmin}_{M_l,\hat{W}_l}\parallel X_lM_l(\hat{W}_l)-X_lW_l \parallel^2_F,$$ 在其中 $M_l,\hat{W}_l$ 分别代表该层的 mask 和经过重构后的参数。后文合适的时候我们会省略下标 $l$。该优化问题是一类 NP-hard 问题，因此 现存所有的方法都依赖于对其解的近似。其中最常用的一种近似方法是先选取某种特定的 $M_l^*(\cdot)$ 之后再求解
$$\mathrm{argmin}_{\hat{W}_l}\parallel X_lM_l^*(\hat{W}_l)-X_lW_l \parallel^2_F,$$
即优先选择 mask，再优化权重；此时问题简化为常规的最小二乘问题。省略下标 $l$ 与 $M_l^*$ 上标 $*$，对 $j\in[C_{out}]$ 取 $i_{M_j}:=(M[:,j] ==1)$（ mask 在第 j 列所保留的位置）并记 $\hat{w}^j=\hat{W}[:,j]$，$\hat{w}^j_{M_j}=\hat{W}[i_M,j]$（W 对第 j 个输出特征所保留的权重），$X_{M_j}=X[:,i_M]$。上述优化有等价于对每个 $j\in[C_{out}]$，求解 $$\mathrm{argmin}_{\hat{w}^j_{M_j}}\parallel X_{M_j}\hat{w}^j_{M_j}-X_l{w}^j \parallel^2_2,$$
<div align="center">
    <img src="./images/IM53.jpg" width=50% height=50%/>
</div>
<br>

由标准最小二乘法的解可得 $\hat{w}^j_{M_j} = (X_{M_j}^TX_{M_j})^{-1}X_{M_j}^T X_l{w}^j$. 求解 $C_{out}$ 次该问题所需的复杂度为 $\mathcal{O}(C_{out}(C_{in}^2\times\mathrm{batch\_size}+C_{in}^3))$，对应 $C_{out}$ 次对 Hessian matrix 的 $H_{M_j}:=X_{M_j}^TX_{M_j}$ 的计算和取逆。如何构造一个可复用或可快速计算的 $H_{M_j}$ 是 SparseGPT 算法实现高效训练的关键。

我们尝试从另一个角度理解上述问题。现在，假设 mask 是未知的，考虑 $X$ 与参数矩阵第 j 列 $\hat{w}^j$ 相乘的结果（我们在下式的推导中先隐去上标 $j$），同时，我们把原先模型在该层对应位置的输出记为 $y:=X{w}^j$。考虑 loss $E(w)=\frac{1}{2}\parallel Xw-y\parallel_2^2$，对于压缩前的模型和参数，显然压缩前的 $w_0$ 是最优的，因为此时 $E(w_0)=0$。对于二次损失函数，这意味着 $\nabla_w^TE(w_0)\equiv 0$。剪枝意味着对 loss 进行了扰动 $\delta w$，记剪枝后的参数为 $w_0+\delta w$，对 $E(w_0+\delta w)$ 展开，
$$E(w_0+\delta w)=E(w_0)+\nabla_w^TE(w_0)\delta w+\frac{1}{2}\delta w^T\nabla_w^2E(w_0)\delta w+\mathcal{O}(\parallel\delta w\parallel_2^3),$$
结合 $\Delta_w^TE(w_0)\equiv 0$ 以及其二次性，得到
$$\delta E(w_0)=\frac{1}{2}\delta w^T\nabla_w^2E(w_0)\delta w.$$
同时不难看出，Hessian matrix $\nabla_w^2E(w_0)=X^TX=:H$。我们进一步考虑在不改变其它参数的情况下，单把 $w$ 的第 $q$ 个参数 $w[q]$ 置零后 $e_q^T\delta w = -w[q]$ 的对于 loss 的最小影响，也即考虑约束优化问题
$$\min_{\delta w}\delta E(w_0)=\frac{1}{2}\delta w^TH\delta w\ \ \mathrm{s.t.}\ \ e_q^T\delta w +w[q]=0.$$
使用拉格朗日乘子法容易求得
$$L_q = \frac{1}{2}\frac{w^2[q]}{H^{-1}_{qq}}\ \ \ ,\ \ \ \delta w=-\frac{w[q]}{H^{-1}_{qq}}H^{-1}e_q$$
前者表示置零参数 $w[q]$ 的最小影响，或 $w[q]$ 的显著性 saliency；后者表示将 $w[q]$ 置零以后像哪个方向优化可以最大程度降低对 loss 的扰动，也即补偿方向。上述思想是 Optimal Brain Surgeon ([OBS](https://www.zhihu.com/search?type=content&q=OBS%20update%20%E5%89%AA%E6%9E%9D)) 剪枝算法的核心。注意到若使用这种方法，我们只需要计算一次 $H$。注意如果总是把上一轮优化结果作为最优结果，即总是假设 $\nabla_w^TE(w_t)\equiv 0$，到后期可能产生较大偏差。而如果损失总是和上轮比，就变成非常标准的贪心算法，最后累积的损失不是最终损失。

根据下图我们简述 SparseGPT 的原理，注意原图的维度（$WX$）和本笔记在此处讨论的维度 $XW$ 是反过来的，正确的顺序应参考下图 2。
<br>
<br>
<div align="center">
    <img src="./images/IM54.png" width=70% height=70%/>
</div>
<br>
<br>
<br>
<div align="center">
    <img src="./images/IM55.png" width=20% height=20%/>
</div>
<br>

（Trivial）算法流程：
- 初始化 index set $U_1:=[C_{in}]$，binary mask $M$（1 代表保留，也即图中黑色部分），$H_{U_1}:=H=X^TX$
- 计算 $B_1:=(H_{U_1})^{-1}$
- 对每次循环 $i\in[C_{in}]$
- 将 $W$ 中 $j_i = (M[i,:]\neq 1)$ 对应位置的 weight 置零；并保持其它位置参数不变，将 $W[i+1:,j_i]$ 中的 weight 依次（按列） 调整：
        - 计算更新位置每一列要对应的补偿方向 $E:=\frac{1}{H_{U_{i}}^{-1}[{1,1}]}H^{-1}_{U_{i}}[:,1]W[i,:]$
        - 将保持不变的列设置补偿方向为 0：$E:=E\cdot\mathrm{diag}(1-M[i,:])$，或者可以通过 broadcast 写作 $E:=(1-M[i,:])\cdot E$
        - 权重更新 $W[i+1,:]:=W[i+1,:]-E[2:,:]$
        - 置零 $W[i,j_i]:=0$
    - 更新 index set $U_{i+1}:=U_{i}-\{i\}$
    - 更新 $(H_{U_{i+1}})^{-1}:= B_{i+1} := (B_i-\frac{1}{B_i[1,1]}B_i[:,1]B_i[1,:])[2:,2:]$（对 $H^{-1}$ LU 分解），证明省略（数学归纳法）
  
上述 $H_{U_{i}}:=H[U_{i},U_{i}]$，这种更新 $(H_{U_{i}})^{-1}$ 的方式整体只有 $\mathcal{O}(C_{in}^3)$，算法整体复杂度仅有 $\mathcal{O}(C_{in}^2\times\mathrm{batch\_size}+C_{in}^2C_{out})$。

原文中使用更复杂的算法还涉及实时更新 mask，见下图
<br>
<div align="center">
    <img src="./images/IM56.png" width=40% height=40%/>
</div>
<br>




<br>

实验结论
- SparseGPT 首次实现了对大模型参数在保留性能的基础上进行快速的大规模压缩；
- 相对来说，越大的模型越容易使用 SparseGPT 进行稀疏化，同样比例的稀疏化，参数越多对原模型性能的影响越小（可能是有过参数化导致的）；
- 结合 SparseGPT 与量化，压缩效果超越了 GPTQ（一种先进的量化手段）；
- 剪枝时，跳过模型的后半段（保留原参数）可以最大程度上保留性能；


<a id="sub-03-08-03"></a>

### 3.8.3 实践：从压缩公式到 AWQ 校准


进入 [多模态压缩实践](practices/mllm-compression-safety/README.md)，阅读 `run.py` 的 quantize：加载独立校准文本 → 捕获语言层激活 → 搜索逐通道缩放/裁剪 → 打包 4 bit 权重 → 保存 processor 和 tokenizer。正式量化需 GPU 和完整模型，不在主教材中自动触发。

先完成等价缩放练习：对于 PyTorch 的线性层 y = x Wᵀ，令 x' = x/s、W' = W·s，未量化时输出不变；量化引入误差后，不同 s 才会影响结果。下面只验证等价式，**不是 AWQ 实现或精度报告**。AWQ 此处的裁剪服务于量化重构，不是训练正则化或梯度裁剪。


In [ ]:
import torch
x, weight = torch.randn(4, 8), torch.randn(3, 8)
scale = torch.rand(8) + 0.5
torch.testing.assert_close(x @ weight.T, (x / scale) @ (weight * scale).T)
print('Equivalent scaling before quantization: passed')


---

<a id="part-03-09"></a>

## 3.9 强化学习基础

> **本节目标**：认识状态、动作、奖励、策略与价值，为理解对齐方法准备基础。

**小节导航**

- [3.9.1 基础概念 Introduction](#sub-03-09-01)
- [3.9.2 策略优化方法 Policy-based Learning](#sub-03-09-02)

[返回章节目录](#chapter-03)


强化学习 Deep Reinforcement Learning

<a id="sub-03-09-01"></a>

### 3.9.1 基础概念 Introduction


<a id="sub-03-09-01-01"></a>

#### 3.9.1.1 术语 Terminologies


强化学习实际上可被视为一种解决 **马尔科夫决策过程 (Markov decision processes, MDP)** 的技术。马尔科夫决策过程是马尔可夫链的推广，不同之处在于添加了 **行动** 和 **奖励** 机制。在每个时间步骤中，随机过程都处于某种 **状态 state**，记为 $s \in \Omega$。决策者可以选择在状态 $s$ 下可用的动作 $a\in \mathbb{A}$，使得该随机过程在下一时间以一定的概率 $P_a(\ s\ ,\ s'\ )=\mathbb{P}(\ S_{t+1}=s'\ |\ S_t=s\ ,\ A_t=a\ )$ 进入新状态 $s'\in \Omega$，并给予 **决策者/智能体 Agent** 相应的 **回馈/奖励 Reward** $R_{a}(\ s\ )$，严格定义如下：

马尔科夫决策过程 (Markov decision processes, MDP) 指的是如下的一个结构 $(\ \Omega\ , \ \mathbb{A}\ ,\  P\ ,\  R\ )$ 其中：
* $\Omega$ is the set of __states__
* $\mathbb{A}$ is a finite set of __actions__
* $P$ is a __transition function__ $P:\mathbb{A} \times (\Omega \times \Omega) \to [\ 0\ ,1\ ]$, for which $P_a(\ s\ ,\ s'\ )$ gives the probability that action $a$ performed in state $s$ will lead to state $s'$
* $R$ is a __reward function__ defined as $R:\mathbb{A} \times \Omega \to \mathbb{R}$

在 $t$ 时刻，决策者根据当前的状态 $S_t=s$ 以一定的概率 $\rho\ (\ a\ |\ s\ )$ 执行相应的行为 $A_t=a$，称这一概率为 **策略函数/策略 policy**：

$$\rho: \mathbb{A} \times \Omega\to [\ 0\ ,1\ ]\ \ s.t.\ \ \rho\ (\ a\ |\ s\ )=\mathbb{P}(\ A_t=a\ |\ S_t=s\ )$$

由于决策者的状态与行为 $S_t$ & $A_t$ 在 $t$ 时刻均为随机变量，此时获得的回报也应当为一个随机变量，故记为 $R_t=R_{A_t}\ (\ S_t\  )$。在给定上述决策过程与一个（初始化）策略函数后，开始运行该决策过程直到其**达成某一终止状态**，比如赢得或输掉游戏，所得到的一条确定的**样本轨迹** $(\ s_t\ , \ a_t\ ,\  r_t\ )$，其中 $r_t=R_{a_t}\ (\ s_t\ )$，称为一个 **Episode**。

<p align="center">
<img src="./images/DRL_01.png" width="800" title="Image." >
</p>



对于决策从开始到终止的一段时间内，在 $t$ 时刻的 **(未来)累计回报 Cumulative Return** 定义为随机变量 $C_t= {\textstyle \sum_{i=0}^{\infty}R_{t+i}}$，考虑到回报的 **时间价值**，定义 **(未来)贴现回报 Discounted Return** 为随机变量 $D_t= {\textstyle \sum_{i=0}^{\infty}\gamma^{i} R_{t+i}}\ \ s.t.\ \ 0\le \gamma \le1$；为了进一步衡量回报，定义过程在给定策略下的 
- **动作-价值函数 Action-Value Function** 为 $Q_{\rho}(\ s_t\ ,\ a_t\ )=\mathbb{E}\ [\ D_t\ | \ S_t=s_t\ ,\ A_t=a_t\ ]$ 表示在状态 $s_t$ 下执行 $a_t$ 的期望回报
- **最优动作-价值函数 Optimal Action-Value Function** 为 $Q^{*}(\ s_t\ ,\ a_t\ )=\max_{\rho}Q_{\rho}(\ s_t\ ,\ a_t\ )$ 表示在状态 $s_t$ 下执行 $a_t$ 的最大期望回报
- **状态-价值函数 State-Value Function** 为 $V_{\rho}(\ s_t\ )=\mathbb{E}_A\ [\ Q_{\rho}(\ s_t\ ,\ A\ )\ ] = \mathbb{E}\ [\ Q_{\rho}\ |\ S_t=s_t\ ]$ 以评价当前状态 $s_t$ 与策略 $\rho$ 的好坏

按照学习对象分类，强化学习大致有以下两种策略：
- **Policy-based Learning 策略优化方法**：学习一个策略函数 $\rho$，Agent根据当前状态与策略函数给定的分布，从中随机抽样做出行动 $a_t \sim  \rho\ (\ \cdot\ |\ s_t\ )$
- **Value-based Learning 价值优化方法**：学习一个最优动作-价值函数 $Q^{*}$，Agent根据当前状态选择最大化 $Q^*$ 的行动 $a_t =\arg\!\max_{a}Q^{*}(\ s_t\ ,\ a_\ )$

[推荐教程](https://www.bilibili.com/video/BV1FM411L7QW/?spm_id_from=333.337.search-card.all.click&vd_source=ff901644057cda4596a72384c16c4fb4)


<a id="sub-03-09-01-02"></a>

#### 3.9.1.2 OpenAI gym (现已更新为 gymnasium)

强化学习常用的Python库为 [**OpenAI gym**](https://github.com/openai/gym)，其中涉及到三类基本任务：
- **Classical Control Problems**：经典控制问题，如控制机械的运动
- **Atari Games**：小游戏
- **MuJoCo (Continuous Control Tasks)**：连续控制问题，如控制机器人行走等

研发者也常用该库测试自己的模型与算法，使用 **gym** 的几个最基本的命令如下 (**import gym**) ：
- <span style="color: blue;">**env = gym.make("FrozenLake-v1")**</span> 创建一个任务对象，此处的训练任务为 "FrozenLake" 游戏
- <span style="color: blue;">**state = env.reset()**</span> 初始化环境，回到原始状态
- <span style="color: blue;">**env.render()**</span> 渲染模型并将任务内容可视化
- <span style="color: blue;">**state, reward, done, \_ , \_= env.step(action)**</span> 按照给定的 action 更新 state ，返回此次更新的 reward ；done 表示 episode 是否结束，action 需要通过训练与计算得到


<a id="sub-03-09-02"></a>

### 3.9.2 策略优化方法 Policy-based Learning


<a id="sub-03-09-02-01"></a>

#### 3.9.2.1 交叉熵法 Cross-Entropy Method CEM




<a id="sub-03-09-02-02"></a>

#### 3.9.2.2 策略梯度法 Policy Gradient Methods



---

<a id="part-03-10"></a>

## 3.10 AI 对齐

> **本节目标**：理解对齐目标、失效模式以及监督学习、强化学习与对齐之间的关系。

**小节导航**

- [3.10.1 The Mission of Alignment](#sub-03-10-01)

[返回章节目录](#chapter-03)


AI 对齐 [AI Alignment](https://www.bilibili.com/video/BV1Pc411d7Fh/?spm_id_from=333.337.search-card.all.click&vd_source=ff901644057cda4596a72384c16c4fb4)
[原文链接](https://arxiv.org/pdf/2310.19852)
- Purpose: Build AI systems that behave in line with human intentions and values.

<a id="sub-03-10-01"></a>

### 3.10.1 The Mission of Alignment


<a id="sub-03-10-01-01"></a>

#### 3.10.1.1 非对齐现象的定义 The Failure Modes of Alignment

**常见的非对齐现象**
- **奖励欺骗 Reward Hacking**: Agent 通过偏好最大化接收到奖励的行动来欺骗系统，即“钻空子”，使这些行动与最初的目标不一致。An agent achieves high rewards by exploiting a poorly defined reward function, leading to strong performance in certain metrics but ultimately falling short of human standards.
  - **规范博弈 Specification Gaming**: 是一种满足了目标的字面规范（literal specification），但没有实现预期结果的现象。
  - **奖励篡改 Reward Tampering**: 模型通过某种途径直接修改其自身的奖励机制。可以视为 Reward Hacking 的一种特例。
- **目标错误泛化 Goal Misgeneralization GMG**: 指学习系统采取有效的手段完成了**预期之外的目标**，这会导致在训练情境中表现良好，但在新的测试情境中表现不佳（在训练情境中错误目标可能和预期目标达到相同的效果）。

**非对齐模型的危害**
- **Power Seeking**: 智能体试图获得控制环境的更普遍的能力（完成任务-保护电源-阻止人类关闭电源）。
- **Manipulation**: 操纵人类达成目的（GPT假扮残疾人士博取人类同情通过验证码）。
- **Deceptive Alignment / Scheming**: 没有对齐的人工智能暂时表现出完成对齐的样子，以欺骗其创造者，避免被关闭或重新训练并获得创造者赋予已对齐人工智能的权力。
- **Collectively Harmful Behaviors**: 人工智能系统有可能采取一些行动，其在孤立的情况下看似无害但在多智能体或社会背景下可能为问题。
- **Violation of Ethics**: 对人类社会的道德、价值观念产生影响。
  
**模型对齐的双刃剑**
- **Situational Awareness**: 使智能体及时意识到当下其所具备和不具备的能力。
- **Broad-scoped Goals**: 使智能体具有全局观和长远规划能力。
- **Mesa-optimize Goals**: 外部优化过程（如机器学习中的训练阶段）会创造出本身就是优化器的人工智能。
- **Access to More Resources**: 使智能体获得更多的资源。


<a id="sub-03-10-01-02"></a>

#### 3.10.1.2 模型对齐的目标 The Objective of Alignment

Concretely, we characterize the objectives of alignment with four principles: **Robustness, Interpretability, Controllability, and Ethicality (RICE)**. 
- **Robustness**: 鲁棒性指人工智能系统在不同场景或对抗压力下运行时的复原力，特别是其目标的正确性以及能力。稳健的人工智能系统应能应对黑天鹅事件和长尾风险，以及各种对抗压力。
- **Interpretability**: 可解释性要求我们能够理解人工智能系统的内在推理，尤其是不透明神经网络的内部运作。可解释性还能让用户和利益相关者了解和理解决策过程，从而实现人类监督。
- **Controllability**: 可控性是确保系统的行动和决策过程始终受人类监督和干预的必要属性。它保证人类干预可以及时纠正系统行为中的任何偏差或错误。
- **Ethicality**: 道德性是指一个系统在决策和行动中坚定不移地维护人类准则和价值观的承诺。它确保系统避免采取违反道德规范或社会习俗的行动，例如对特定群体表现出偏见，对个人造成伤害，以及在汇总偏好时缺乏多样性或平等性。

---

<a id="part-03-11"></a>

## 3.11 推理与评估

> **本节目标**：通过推理任务、能力评估与安全评估检验模型，并完成多模态评估练习。

**小节导航**

- [3.11.1 Chain-of-Thought Prompting Elicits Reasoning in Large Language Models](#sub-03-11-01)
- [3.11.2 Reasoning with Large Language Models, a Survey (16 Jul 2024)](#sub-03-11-02)
- [3.11.3 LLM Agent Benchmark List](#sub-03-11-03)
- [3.11.4 A Survey of Safety and Trustworthiness of Large Language Models through the Lens of Verification and Validation, 2023](#sub-03-11-04)
- [3.11.5 How Numerical Precision Affects Mathematical Reasoning Capabilities of LLMs](#sub-03-11-05)
- [3.11.6 实践：评估覆盖率与分母](#sub-03-11-06)

[返回章节目录](#chapter-03)


LLM Reasoning and Evaluation

<a id="sub-03-11-01"></a>

### 3.11.1 [Chain-of-Thought Prompting Elicits Reasoning in Large Language Models](https://arxiv.org/abs/2201.11903)

[视频介绍](https://www.bilibili.com/video/BV1t8411e7Ug/?spm_id_from=333.337.search-card.all.click&vd_source=ff901644057cda4596a72384c16c4fb4) 见此处。This paper explores how generating a **chain of thought (CoT)** - a series of intermediate reasoning steps - can significantly improve the ability of large language models to perform complex reasoning tasks. The authors show that chain-of-thought prompting, where a few chain of thought demonstrations are provided as exemplars, enables large language models to tackle arithmetic, commonsense, and symbolic reasoning tasks. Experiments on three large language models demonstrate that chain-of-thought prompting outperforms standard prompting, sometimes to a striking degree, and can achieve state-of-the-art performance on benchmarks like the GSM8K math word problem dataset.

- LLMs successfully perform **system-1 tasks**, which are **done quickly and intuitively by humans**, such as sentiment analysis and topic classification.
- LLMs struggle on **system-2 tasks**, which requires slow and deliberate thinking (often with multiple steps), and includes
logical, mathematical, and commonsense reasoning tasks, among others.
- In this paper, we explore **chain of thought (CoT)** prompting as a method for **improving** the ability of language models to perform **reasoning (system-2) tasks** (比如在 [few-shot learning](https://blog.csdn.net/zcyzcyjava/article/details/127006287) 中给出解题步骤).
<br>
<br>
<div align="center">
    <img src="./images/IM67.png" width=25% height=25%/>
    <img src="./images/IM68.png" width=60% height=60%/>
</div>

- Extensions: [Zero-Shot CoT: Large Language Models are Zero-Shot Reasoners](https://arxiv.org/pdf/2205.11916), [Auto CoT: Automatic Chain of Thought Prompting in Large Language Models](https://arxiv.org/abs/2210.03493)


<a id="sub-03-11-02"></a>

### 3.11.2 [Reasoning with Large Language Models, a Survey (16 Jul 2024)](https://arxiv.org/html/2407.11511v1#S6)


<a id="sub-03-11-02-01"></a>

#### 3.11.2.1 Basic Concepts

- Transformer-based generative language models whose size is beyond hundreds of billions parameters are not only very good at language generation, they also enable new type of machine learning, called **[in-context learning](https://cloud.tencent.com/developer/article/2303328)**, which is also known as **prompt-based learning** and occurs only in LLMs beyond a certain size (hundreds of billions of parameters) that are sufficiently rich. In-context learning is **inference time (推理阶段进行)**, **prompt-based**, **few-shot learning**, where model **parameters are not trained or fine-tuned**. In-Context Learning 最初是在原始 GPT-3 论文中作为一种大语言模型学习任务的方式而被推广的，能够直接让语言模型根据给定的几个实例理解任务，并给出问题答案；本质上，它相当于使用训练完好的语言模型估计给定示例条件下的条件概率分布模型。在 In-Context Learning 里，给语言模型一个提示（prompt），该提示是一个由输入输出对组成的列表，这些输入输出对用来描述一个任务。在提示的末尾，有一个测试输入，并让语言模型仅通过以提示为条件来预测下一个标记。
  
- **System-1 tasks**, such as associative language tasks, are easily solved by LLMs with in-context learning. On the other hand, **System-2 tasks**, such as grade school math word problems, are more difficult for LLMs. To solve math word problems, we need to break down the problem into multiple reasoning steps. Spurred on by the impressive performance of System 1 tasks, much research has focused on understanding the reason for the poor performance of LLMs on System 2 tasks and how it can be improved. Among this research, the **Chain-of-thought (CoT)** experiment stands out. This work, and subsequently Kojima et al. (**Zero-Shot CoT**), showed that adding a simple instruction to the prompts, **"Let’s think step by step"**, can provoke an LLM to perform the required intermediate reasoning steps, achieving a surprising jump in performance. The Chain-of-thought paper is a breakthrough in the field of reasoning with LLMs. Much exciting work has been published that builds on this work.


<a id="sub-03-11-02-02"></a>

#### 3.11.2.2 Benchmark

Progress in artificial intelligence is measured by benchmarks. Benchmarks define the goal that researchers aim to achieve in their experiments. In natural language processing, a wide array of benchmarks exists to measure progress, such as

- CommonsenseQA [Talmor et al., 2018] on question answering
- LAMBADA [Paperno et al., 2016] on word prediction
- WMT’22 [Kocmi et al., 2022] on translation
- GLUE [Wang et al., 2018, 2019] on language understanding
- Xsum [Narayan et al., 2018] on text summarization

For reasoning ability (Math)
- **MAWPS benchmark [Koncel-Kedziorski et al., 2016]**: The Math Word Problem Repository (MAWPS) allows for constructing datasets with particular characteristics by selecting different categories of problems. The dataset consists of 3320 problems. An example is: **"Problem: Rachel bought two coloring books. One had 23 pictures and the other had 32. After one week she had colored 44 of the pictures. How many pictures does she still have to color? Answer: 55 − 44 = 11."** The baseline performance of **GPT-3 175B is 72.7% accuracy**. The performance of **Chain-of-thought is 87.1% accuracy**.

- **AQuA [Ling et al., 2017]**: The Algebraic Question Answering dataset is a **large dataset of 100,949 questions, answers, and rationales**. The dataset is based on a combination of a smaller seed dataset and crowdsourcing. An example question is: **"Question: Two trains running in opposite directions cross a man standing on the platform in 27 seconds and 17 seconds respectively and they cross each other in 23 seconds. The ratio of their speeds is: Options: A) 3/7 B) 3/2 C) 3/88 D) 3/8 E) 2/2 Answer: B."** The baseline performance of **GPT-3 175B is 24.8% accuracy**. The performance of **Chain-of-thought is 35.8% accuracy**.

- **GSM8K [Cobbe et al., 2021]**: To test reasoning skills, the Grade School Math problem dataset (GSM8K) was developed for testing LLMs. It consists of 8500 human-written math problems. Language models struggled to perform well on this dataset (**pre Chain-of-thought**). An example of a math word task is: **"Problem: Beth bakes 4, two dozen batches of cookies in a week. If these cookies are shared amongst 16 people equally, how many cookies does each person consume? Answer: 4 × 2 × 12/16 = 6."** The baseline performance of **GPT-3 175B is 15.6% accuracy**. In comparison, the performance of **Chain-of-thought is 46.9% accuracy**.

- **ASDiv [Miao et al., 2021]**: The Academia Sinica Diverse MWP Dataset (ASDiv) is designed for high diversity in **problem types, formats and difficulty levels**. It consists of 2305 problems. An example problem is: **"Problem: A sandwich is priced at 0.75. A cup of pudding is priced at 0.25. Tim bought 2 sandwiches and 4 cups of pudding. How much money should Tim pay? Answer: 0.75 × 2 + 0.25 × 4 = 2.5."** The baseline performance of **GPT-3 175B is 70.3% accuracy**. The performance of **Chain-of-thought is 71.3% accuracy**.

- **SVAMP [Patel et al., 2021]**: The Simple Variations on Arithmetic Math word Problems dataset (SVAMP) consists of 1000 problems from variations of ASDiv-a and MAWPS. The baseline performance of **GPT-3 175B is 65.7% accuracy**. In comparison, the performance of **Chain-of-thought is 68.9% accuracy**.


<a id="sub-03-11-02-03"></a>

#### 3.11.2.3 Reasoning Taxonomy

- **Prompt for Step Generation**: The problem in the prompt must be split into substeps. This can be achieved with a problem-specific prompt that contains elements of the problem, such as: “First calculate how many marbles Mary had originally, then how many her friend had, and finally how many they had together.” In general, it is possible to prompt an LLM to fill in the blanks in a step-by-step fashion. There are three main approaches for generating the step-by-step prompt. The prompt may be
    - **(1) handcrafted for the problem by the researchers (hand-written prompt)**
    - **(2) the prompt or prompts may come from a source that is external to the model, such as another model or a dataset (prompt using external knowledge)**
    - **(3) the model itself can be prompted to generate a (series of) prompt(s) to analyze the problem (model-generated prompt)**.
  
<br>

- **Result Evaluation**: After the prompt has been generated and the model has answered it, the next step in the reasoning pipeline is to evaluate the answer. The steps may be evaluated by
    - **(1) the model itself (self-assessment), e.g., self-verification and self-consistency.**
    - **(2) an external program can be used to evaluate the steps, e.g., Program-aided-language**. For example, an external interpreter or compiler can be used to check the validity of the outcome.
    - **(3) an external model can be used, LLM or otherwise**. For example, in robotics, an external physics model can determine if certain actions are physically possible (external model validation). All approaches we considered evaluate the output of the model and generate corrective data. That data is then added to the training pipeline (through **fine-tuning** or **data augmentation**), and the model is subsequently finetuned.

<br>

- **Reasoning Control**: This stage controls how many sub-steps are generated, and how deep into the future the reasoning chain is generated. There are three main approaches:
    - **(1) greedy selection, which generates a step and then follows it**; examples include Chain-of-thought, Auto-CoT, and Zero-shot CoT.
    - **(2) ensemble strategy, which generates a set of possible next steps**; examples include Self-consistency, Self-verification, Chain-of-experts, PAL, and MathPrompter.
    - **(3) external control algorithms**, such as a full **tree-shaped search** or **reinforcement learning** approaches; examples include Tree-of-thoughts, Buffer-of-thoughts, Beam-search, ReAct, Reflexion, and Voyager.

<br>
<br>
<div align="center">
    <img src="./images/IM69.png" width=60% height=60%/>
</div>


<a id="sub-03-11-02-04"></a>

#### 3.11.2.4 Scaling

The emergent abilities of LLMs have prompted research into how reasoning capabilities can be transferred to smaller language models. Scaling laws of LLMs are an active area of study, see for example Kaplan et al. [2020], Henighan et al. [2020], Hoffmann et al. [2022]. Comprehensive surveys on knowledge distillation are Xu et al. [2024], Gu et al. [2023]. For reasoning specifically, Magister et al. [2022] have studied reasoning in small language models, using a student model that learns from a teacher model, by finetuning. Another study related to Self-taught-reasoner [Li et al., 2022a] focuses on explanation in small language models, achieving similar results. Other works focus on prompt distillation for retrieval Dai et al. [2022], recommendation [Li et al., 2023], distillation to embodied agents of Chain-of-thought reasoning [Choi et al.], and distillation of LLM graph reasoning [Zhang et al., 2024]. Distillation of reasoning to smaller models can work surprisingly well in situations with more explicit instructions. Distillation is also proposed for bringing results of System 2 rea-soning to System 1 Yu et al. [2024], which brings us to metacognition.




<a id="sub-03-11-03"></a>

### 3.11.3 [LLM Agent Benchmark List](https://github.com/zhangxjohn/LLM-Agent-Benchmark-List?tab=readme-ov-file)


<a id="sub-03-11-03-01"></a>

#### 3.11.3.1 [A Survey of Reasoning with Foundation Models, 2024](https://arxiv.org/pdf/2312.11562)

In this section, [we](https://arxiv.org/pdf/2312.11562) provide a concise overview of various reasoning tasks, as the Figures below show. Here, we present distinct categories of reasoning approaches and tasks:
- Commonsense Reasoning: Exploring the capacity to infer and apply everyday, intuitive knowledge. 
- Mathematical Reasoning: Focusing on the ability to solve mathematical problems and derive logical conclusions.
- Logical Reasoning: Examining the process of drawing inferences and making decisions based on formal logic.
- Causal Reasoning: Investigating the understanding of cause-and-effect relationships and their implications.
- Multimodal Reasoning: Involving reasoning across multiple data modalities, such as text, images, and sensory information.
- Visual Reasoning: Focusing on tasks that require the interpretation and manipulation of visual data.
- Embodied Reasoning: Exploring reasoning in the context of embodied agents interacting with their environment.
- Other Reasoning Tasks: The discussion of reasoning extends across various contexts, including conceptual frameworks, such as
    - theory of mind ToM
    - weather prediction
    - abstract reasoning
    - defeasible reasoning
    - medical reasoning
    - bioinformatic reasoning

<br>
<br>
<div align="center">
    <img src="./images/IM70.png" width=60% height=60%/>
</div>

<br>
<br>
<div align="center">
    <img src="./images/IM71.png" width=60% height=60%/>
</div>

Benchmarks, datasets, and metrics play a crucial role in evaluating and advancing reasoning capabilities in various domains, driving innovation, and fostering the development of more capable and reliable reasoning systems. These resources provide standardized frameworks and tasks that enable researchers and developers to objectively assess the performance of reasoning models and compare different approaches. Representative datasets are summarized in Table 8 and 9.

<br>
<br>
<div align="center">
    <img src="./images/IM72.png" width=40% height=40%/>
    <img src="./images/IM73.png" width=40% height=40%/>
</div>


Other benchmarks for each task:
- Commonsense Reasoning: **CQA** (Talmor et al., 2019) and **CoS-E** (Rajani et al., 2019)
- Mathematical Reasoning: **NPHardEval** (based on problem complexity)

Regarding Safety
- We believe that **the common sense** and **world knowledge** inherited from foundation models could unleash the substantial effectiveness of algorithms onboard to handle corner cases and enhance explainability and safety. Below, we survey this emergent topic from two perspectives.


<a id="sub-03-11-03-02"></a>

#### 3.11.3.2 [A Survey on Evaluation of Large Language Models, 2023](https://arxiv.org/pdf/2307.03109)

<br>
<br>
<div align="center">
    <img src="./images/IM77.png" width=70% height=70%/>
</div>


**NLP**

- Reasoning： Currently, the evaluation of reasoning tasks can be broadly categorized into **mathematical reasoning**, **commonsense reasoning**, **logical reasoning**, and **domain-specific reasoning**.
- **Factuality (hallucination)**：Factuality in the context of LLMs refers to the extent to which the information or answers provided by the model align with real-world truths and verifiable facts. Evaluating factuality is of great importance in order to trust and efficiently use these models. This includes the ability of these models to maintain consistency with known facts, avoid generating misleading or false information (known as “factual hallucination"), and effectively learn and recall factual knowledge.
  - Wang et al. [204] evaluated the internal knowledge capabilities of several large models by examining their ability to answer open questions based on the **Natural Questions** [98] and **TriviaQA** [88] datasets with human assessment.
    - Honovich et al. [74] present **TRUE**: a comprehensive survey and assessment of factual consistency evaluation methods, covering various metrics, tasks and datasets. 
    - Pezeshkpour [156] proposed a novel metric, based on information theory, to assess the inclusion of specific knowledge in LLMs.
    - Gekhman et al. [55] improved the method for evaluating fact consistency in summarization tasks. 
    - Manakul et al. [133] proposed three formulas (**BERTScore [249], MQAG [134] and n-gram**) to evaluate factuality.
    - Min et al. [138] broke down text generated by LLMs into individual “atomic" facts, which were then evaluated for correctness. The **FActScore** is used to measure the performance of estimators through the calculation of F1 scores.
    - **Lin et al. [119]** introduced the <span style="color:red;">**TruthfulQA dataset**</span>, designed to cause models to make mistakes. The findings suggest that simply scaling up model sizes may not necessarily improve their truthfulness. This dataset has become widely used for evaluating the factuality of LLMs [89, 146, 192, 220].

<br>


**Robustness, Ethic, Bias, and Trustworthiness**

  - **Robustness**: Robustness studies the stability of a system when facing unexpected inputs. **Out-of-distribution (OOD)** [207] and **adversarial robustness** are two popular research topics for robustness.
    - <span style="color:red;">**[Wang et al.](https://www.kuxai.com/article/941)**</span> [206] is an early work that evaluated ChatGPT and other LLMs from both the adversarial and OOD perspectives using existing benchmarks such as **AdvGLUE [203], ANLI [140], and DDXPlus [41] datasets**.
    - Zhuo et al. [267] evaluated the robustness of semantic parsing.
    - Yang et al. [234] evaluated OOD robustness by extending the GLUE [200] dataset. The results of this study emphasize the potential risks to the overall system security when manipulating visual input.
    - Zhao et al. [258] evaluated LLMs on visual input and transferred them to other visual-linguistic models, revealing the vulnerability of visual input.
    - Li et al. [111] provided an overview of OOD evaluation for language models: adversarial robustness, domain generalization, and dataset biases. 
    - Liu et al. [123] introduced a large-scale robust visual instruction dataset to enhance the performance of large-scale multi-modal models in handling relevant images and human instructions.
    - **<span style="color:red;">Zhu et al.</span> [264]** evaluated **adversarial text attacks** at multiple levels by proposing a unified benchmark called **PromptBench**. The results showed that contemporary LLMs are vulnerable to adversarial prompts.
    - **<span style="color:red;">Wang et al.</span> [201]** introduced **AdvGLUE++ benchmark** data for assessing adversarial robustness and scrutinized machine ethics via jailbreaking system prompts.
    
  <br>
  
  - **Ethic and bias**: LLMs have been found to internalize, spread, and potentially magnify harmful information existing in the crawled training corpora, usually, toxic languages, like offensiveness, hate speech, and insults [53], as well as social biases like stereotypes.
      - Zhuo et al. [266] used conventional testing sets and metrics [37, 53, 153] to perform a systematic evaluation of ChatGPT’s toxicity and social bias.
      - **<span style="color:red;">Deshpande et al.</span>** [35] observed increased **generated toxicity** up to 6x with **REALTOXICITYPROMPTS dataset**. 
      - Ferrara [42] investigated the underlying mechanisms of biases potentially produced by ChatGPT.
      - [65, 167] assessed LLMs with political tendencies and personality traits based on questionnaires like the Political Compass Test and MBTI test.
      - [69] reveals that existing LMs have potential in ethical judgment.
      - [256] proposes a Chinese conversational bias evaluation dataset CHBias, and explores debiasing methods.
      - [209] discovered a systematic bias in the assessment of GPT-4 alignment; [16] discovered bias on cultural values in ChatGPT.
      - **[Wang et al.](https://arxiv.org/pdf/2306.11698) [201]** (2023) incorporated an evaluation dataset specifically aimed at evaluating  toxicity, stereotype bias, adversarial robustness, out-of-distribution robustness, robustness on adversarial demonstrations, privacy, machine ethics, and fairness; see **[DecodingTrust](https://decodingtrust.github.io/)**.
    
  <br>
  
  - **Trustworthiness**: Some work focuses on other trustworthiness problems in addition to robustness and ethics.
      - **<span style="color:red;">Wang et al. [201]** offered a multifaceted exploration of trustworthiness vulnerabilities in the GPT models, especially GPT-3.5 and GPT-4 (DecodingTrust). They revealed that while GPT-4 often showcases improved trustworthiness over GPT-3.5 in standard evaluations, it is simultaneously more susceptible to attacks.
      - Hagendorff and Fabi [62] evaluate LLMs with enhanced cognitive abilities, and found that these models can avoid common human intuitions and cognitive errors. 
      - [228] found that the consistency of judgment in LLMs diminishes notably when faced with disruptions.
      - **Liu et al. [123]** introduced a comprehensive and robust large-scale visual instruction dataset: **LRV-Instruction** for the evaluation of illusions in **large-scale visual model**. 
      - Li et al. [113] reveal through experiments that the distribution of objects in visual instructions significantly impacts object illusions in LVLMs, and introduced a polling-based query method, known as POPE to improve the evaluation of object illusions in LVLMs.


<br>


**Benchmarks**

- **General language task**
    - **HELM [114]** provides a comprehensive assessment of LLMs across various aspects such as language understanding, generation, coherence, context sensitivity, **common-sense reasoning**, and domain-specific knowledge. 
    - **LLMEval2 [252]** encompasses a wide range of capability evaluations.
    - **BIG-bench [182]** introduces a diverse collection of 204 challenging tasks contributed by 450 authors from 132 institutions to evaluate LMs beyond their existing capacities, covering various domains such as math, childhood development, linguistics, biology, **common-sense reasoning (e.g., StrategyQA), social bias**, physics, software development, etc.
    - **KoLA [236]** focuses on assessing language models’ **comprehension** and using semantic knowledge for inference, serving as an important benchmark for evaluating the depth of language understanding and reasoning in language models.
    - **DynaBench [94]** supports dynamic benchmark testing, exploring new research directions, including the effects of closed-loop integration, distributional shift characteristics, annotator efficiency, influence of expert annotators, and **model robustness to adversarial attacks (hate speech)** in interactive settings.
    - **GLUE-X [234]** is a novel attempt to create a unified benchmark to evaluate the OOD robustness of NLP models. 
    - **BOSS [239]** is a benchmark collection for assessing OOD robustness in natural language processing tasks.
    - **PromptBench [264]** evaluates the robustness of LLMS on adversarial prompts.

<br>

- **Specific downstream task**
    - **MultiMedQA [177]** is a medical QA benchmark that focuses on **medical** examinations, medical research, and consumer healthcare questions to evaluate the performance of LLMs in terms of clinical knowledge and QA abilities.
    - **FRESHQA [198]** assesses the ability of LLMs in dynamic QA about current world knowledge by incorporating relevant and current information retrieved from search engines into prompts to discover **hallucinations**.
    - **TRUSTGPT [79]** addresses critical ethical dimensions, including **toxicity, bias, and value alignment**, within the context of LLMs.
    - **EmotionBench [76]** evaluates the simulation of human **emotional reactions** by LLMs.
    - **SafetyBench [254]** is a benchmark specifically designed to test the **security performance** of a range of popular Chinese and English LLMs.
    - **Choice-75 [75]** evaluates the daily **decision-making** capabilities of intelligent systems.
    - **CELLO [66]** assesses LLMs’ aptitude in **understanding complex instructions**.
    - **CUAD [71]** is an expert-annotated, domain-specific **legal contract review dataset** that presents a challenging research benchmark and potential for enhancing deep learning models’ performance in contract understanding tasks.
    - **CVALUES [230]** introduces a humanistic evaluation benchmark to assess the alignment of LLMs with **safety and responsibility** standards.
    - **UHGEval [116]** is a benchmark designed to evaluate Chinese LLMs in text generation without being constrained by **hallucination**-related limitations.

<div align="center">
    <img src="./images/IM76.png" width=70% height=70%/>
</div>



<a id="sub-03-11-04"></a>

### 3.11.4 [A Survey of Safety and Trustworthiness of Large Language Models through the Lens of Verification and Validation, 2023](https://arxiv.org/pdf/2305.11391)


<div align="center">
    <img src="./images/IM78.png" width=50% height=50%/>
    <img src="./images/IM79.png" width=50% height=50%/>
</div>


**Vulnerabilities, Attacks, and Limitations**
- ***Unintended Bugs*** are made by the developers unconsciously but have serious consequences.
    -  **Incidental Exposure of User Information**: In addition to the above attacks that an attacker actively initiates, ChatGPT was reported [6] to have a “chat history” bug that enabled the users to see from their Chat-GPT sidebar's previous chat histories from other users.
    -  **Bias and Discrimination**: LLMs are trained from data, which may include bias and discrimination. 

<br>

- ***Inherent issues*** are vulnerabilities that cannot be readily solved by the LLMs themselves but can be gradually improved with more data and novel training methods.
    -  **Performance Weaknesses**: Aspects in which LLMs have not reached human-level intelligence. Performance issues related to the correctness of the outputs include at least the following two categories:
        -  *factual errors/hallucination*:  where the output of an LLM contradicts the truth.
        -  *reasoning errors*: where LLMs fail to provide correct answers in calculations or logic/causal reasoning questions.
    -  **sustainability Issues**: The training and daily execution of LLMs can have non-negligible sustainability implications due to their huge sizes.
    -  **Trustworthiness and Responsibility**: Generally, these can be grouped into two sub-classes concerning the training data and the final model：
        -  *training data*: There are issues around the copyright, quality, and privacy of the training data.
        -  *final model*: significant concerns include, e.g., LLMs’ capability of independent and conscious thinking [144], LLMs’ ability to be used to mimic human output such as academic works [195], use of LLMs to engage scammers, use of LLMs in generating malware [127, 236, 59], etc.

<br>

- ***Intended Attacks*** are initiated by malicious attackers, which attempt to implement their goals by attacking certain stages in the LLMs lifecycle.
    -  **Unauthorised Disclosure and Privacy Concerns**: For LLMs, it is known that by utilising [prompt injection](https://zhuanlan.zhihu.com/p/689786302), it is possible to disclose the sensitive information of LLMs.
    -  **Robustness Gaps (Adversarial Attack)**: An *adversarial attack* is an intentional effort to undermine the functionality of a model by injecting distorted inputs that lead to the model’s failure. [282] suggests that ChatGPT is vulnerable to adversarial examples, including the single-character change. Moreover, [316] extensively evaluates the adversarial robustness of ChatGPT in natural language understanding tasks using the adversarial datasets **AdvGLUE** [313] and **ANLI** [239]. 
    -  **Backdoor Attacks**: Backdoor attacks is to inject malicious knowledge into the LLMs through either the training of poisoning data or modification of model parameters. Such injections should **not compromise the model performance** and must be bypassed by human inspection. The backdoor will be activated only when input prompts to LLMs contain the trigger, and the compromised LLMs will behave maliciously as the attacker expected. 
    -  **Poisoning and Disinformation**: Poisoning attacks attempt to manipulate some of the training data, which might lead the model to generate wrong or biased outputs.

<br>

**Benchmarks**
- In [316], AdvGLUE and ANLI benchmark datasets are used to assess adversarial robustness, and Flipkart review and DDXPlus medical diagnosis datasets are used to evaluate OOD evaluation. 
- In [293] ([here](https://arxiv.org/pdf/2304.10436)), eight kinds of typical safety scenarios and six types of more challenging instruction attacks are used to expose the safety issues of Chinese LLMs.
- In [118], the **GHOSTS** dataset is used to evaluate the mathematical capability of ChatGPT.
- Note that regarding the LLMs as a software as a service, rather than previous deep learning models, it becomes imperative to incorporate lifelong time assessment. 


<a id="sub-03-11-05"></a>

### 3.11.5 [How Numerical Precision Affects Mathematical Reasoning Capabilities of LLMs](https://arxiv.org/pdf/2410.13857)




<a id="sub-03-11-06"></a>

### 3.11.6 实践：评估覆盖率与分母


继续 [多模态安全评估](practices/mllm-compression-safety/README.md)。固定图像、问题 ID、生成设置，分别运行浮点和 AWQ 模型；再独立标注，不把缺失/失败的评审算作安全。

`metrics.summarize` 遍历问题和模型，分别统计 safe、unsafe、unjudged，报告 coverage = judged / total 和 unsafe_rate = unsafe / judged。覆盖率不同的两组不能直接比较为完整 benchmark 分数；没有标注时指标为 None。下面是合成标签练习，不是实际模型结果。


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('practices/mllm-compression-safety').resolve()))
from metrics import summarize
toy = {
    '0': {'ans': {'demo': {'is_safe(gpt)': 'safe'}}},
    '1': {'ans': {'demo': {'is_safe(gpt)': 'unsafe'}}},
    '2': {'ans': {'demo': {}}},
}
result = summarize(toy)['demo']
assert result['unjudged'] == 1 and result['unsafe_rate'] == 0.5
print(result)
